# Grand-average coherence

The main plots concerning the "grand-average coherence". The notebook has a setting "PRIMARY_REFERENCE" with the value "avg", "linked_mastoids", or "FCz". The figures will be generated based on the selected reference.

See /home/gu08wuvu/eeg-envelope-coherence-neuroimage/exploration/2_grand_average_coherence.ipynb for guiding code.

## Figures produced:

1. Grand average coherence: weighted-average of complex-coherence over all trials. The range of the 500 null profiles is shaded in grey. Topomaps of coherence within predefined bands are shown, with channel markers indicating statistical significance.  The rates of the speech units, and the pitch histograms, are plotted in a lower panel. 

2. Three stacked plots, each similar to panel a above. The plots show the coherence spectrum, topographies, and null distribution, for each of the three possible references.

3. For all of the three references, a topomap of coherence in the low-delta band, next to a plot of lateralisation plot. The recipe for lateralisation is as so (note that the cap is symmetric):

    - channels on the midline (i.e. with names ending in z) have lateralisation 0.
    - even channels are on the right, and are paired with odd channels from the left. E.g. TP9 is paired with TP10, C3 is paired with C4.
    - lateralisation index for each channel is therefore defined as (left-right)/(left+right)
    - evaluate the LI for every subject; statistically test the distribution against zero using a non-parametric test. correct for multiple comparisons (don't double count channel pairings though).
    - indicate statistical with channel markers.

4. A plot like Figure_2_subjects_bands from the old notebook.

5. A plot like figure_2_subjects

6. A plot like Figure_2_channels from the old notebook.

## Setup

`PRIMARY_REFERENCE` selects the EEG reference this notebook's single-reference
figures are built from. Every cache and every figure name follows from it, so one
reference can never read another's cache or overwrite its figures.

Figures 2 and 3 of the list above are comparisons *across* references and load all
three regardless; this setting governs figures 1, 4, 5 and 6.

Inputs are the per-stimulus null files from `get_null_gammatone_coherence.py`
(`null_coherence_dir/gammatone_null_by_stimulus`), which carry the real and the null
coherence together with the durations that weight them, and the
`scripts/speech_units_pitch/` outputs in `speech_units_dir`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from mne import pick_info
from mne.io import read_raw_fif
from mne.viz import plot_topomap
from scipy.ndimage import median_filter
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "figures" else Path.cwd()
CONFIG = json.loads((ROOT / "src" / "package_data" / "config.json").read_text())

# --- which EEG reference the single-reference figures are built from ----------
PRIMARY_REFERENCE = "avg"  # "avg" | "linked_mastoids" | "FCz"
REFERENCES = ("avg", "linked_mastoids", "FCz")

sys.path.insert(0, str(ROOT / "src"))
from coherence.pipeline import reference_channel_names, resolve_reference_dirs

OUT_DIR = ROOT / "figures"
# One cache directory per reference, so a run under one name can never be read back
# under another. Figure 1 of this repo writes its own caches alongside these.
CACHE_ROOT = OUT_DIR / "figure_data"


def null_dir(reference):
    """That reference's per-stimulus pooled-null directory."""
    return resolve_reference_dirs(CONFIG, reference, ROOT)[1] / "gammatone_null_by_stimulus"


def cache_dir(reference):
    path = CACHE_ROOT / reference
    path.mkdir(parents=True, exist_ok=True)
    return path


SPEECH_UNITS_DIR = Path(CONFIG["speech_units_dir"])
PITCH_DIR = SPEECH_UNITS_DIR / "pitch"
EEG_DIR = Path(CONFIG["eeg_dir"])

# Sensor positions for the topographies. get_null_gammatone_coherence.py builds its EEG
# through pipeline.load_eeg, which does raw.pick("eeg"), so the channel axis of the saved
# coherence is exactly this info's channel order with the mastoids already dropped.
TEMPLATE_EEG = EEG_DIR / "sub-001_-_ses-001_-_trial-001_-_cond-natural_-_audio01-audiobook_5_1_-_eeg.fif"
RAW_INFO = read_raw_fif(TEMPLATE_EEG, preload=False, verbose="ERROR").pick("eeg").info


def info_for(reference):
    """`RAW_INFO` narrowed to the channel axis the pooled files carry for `reference`.

    Not the raw file's axis: FCz is dropped once referenced to itself, leaving 63. Every
    topography and sensor average then indexes the same channels the data actually has.
    """
    names = reference_channel_names(TEMPLATE_EEG, reference)
    if list(RAW_INFO.ch_names) == names:
        return RAW_INFO
    return pick_info(RAW_INFO, [RAW_INFO.ch_names.index(name) for name in names])


INFO = info_for(PRIMARY_REFERENCE)

# --- house style, shared with figure 1 ---------------------------------------
DEFAULT_FONTSIZE = 10
TICK_FONTSIZE = 8
SMALL_FONTSIZE = 6.5
PANEL_LABEL_FONTSIZE = 14
FIGSIZE = (6.27, 3 * 5.894 / 4)

INK, MUTED, HAIRLINE = "#1b1b1a", "#6b6b68", "#d5d5d1"
NULL_FILL = "#b9b9b4"

plt.rcParams.update({
    "font.size": TICK_FONTSIZE,
    "axes.labelsize": DEFAULT_FONTSIZE,
    "axes.titlesize": DEFAULT_FONTSIZE,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "axes.linewidth": 0.8,
    "figure.dpi": 130,
    # TrueType, not matplotlib's default Type 3, which publishers flag and which is
    # not reliably searchable. Has to be set before anything is drawn.
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def bare(ax):
    """Recessive chrome: drop the top/right rules."""
    ax.spines[["top", "right"]].set_visible(False)
    return ax


print(f"primary reference: {PRIMARY_REFERENCE} ({len(INFO['ch_names'])} channels)")
for reference in REFERENCES:
    print(f"  {reference:16s} {len(sorted(null_dir(reference).glob('*.npz'))):3d} stimulus null files")

## Pooling the trials

Each of the 69 per-stimulus files holds that stimulus's duration-weighted real
coherence and its 500 null draws, all `(F, n_eeg)` planes on the same 0.2 Hz grid.
Pooling is one more duration-weighted average across stimuli, taken on the
**complex** coherence before any magnitude is formed -- the same order as within a
stimulus, so the grand average is exactly the average over all 563 trials.

Draw *i* of the pooled null is the average of every stimulus's draw *i*, so the 500
pooled draws are 500 realisations of the *pooled* statistic, which is what the real
grand average has to be judged against: both for the spectrum in panel (a) and for
the per-band test below.

Reading one reference's 69 files is ~43 GB, so the reduction is cached. Delete the
`.npz` under `figures/figure_data/<reference>/` to force a rebuild.

In [ ]:
# Five pre-defined bands. The frequency each topography is anchored to is not fixed
# here: it is fitted from the spectrum further down, once there is a spectrum to fit.
BASE_BANDS = [(0.2, 2.0), (4.0, 15.0), (30.0, 60.0), (80.0, 120.0), (140.0, 250.0)]

# Bands carried by one reference only. Linked mastoids shows a bump near 23 Hz that the
# other two do not, so it gets a band of its own rather than forcing one on all three.
EXTRA_BANDS = {"linked_mastoids": [(20.0, 26.0)]}


def bands_for(reference):
    """The bands this reference is analysed over, in frequency order."""
    return np.array(sorted(BASE_BANDS + EXTRA_BANDS.get(reference, [])))


# The primary reference's bands, under the name the single-reference cells below use.
BANDS = bands_for(PRIMARY_REFERENCE)

# Mains hum and its harmonics are excluded from the band averages: they are a shared
# artefact of both signals, so they inflate the real coherence without being tracking.
LINE_FREQ, NOTCH_HALFWIDTH = 50.0, 2.0

ALPHA = 0.05


def band_bin_masks(frequencies, bands=None):
    """One boolean mask per band, mains harmonics excluded, plus their union."""
    bands = BANDS if bands is None else bands
    harmonics = np.arange(LINE_FREQ, frequencies[-1] + LINE_FREQ, LINE_FREQ)
    line = (np.abs(frequencies[:, None] - harmonics) <= NOTCH_HALFWIDTH).any(1)
    masks = [(frequencies >= low) & (frequencies <= high) & ~line for low, high in bands]
    return masks, np.flatnonzero(np.any(masks, axis=0))


def pool_null_files(paths, bands):
    """Reduce ~43 GB of per-stimulus null files to the few arrays the figure needs.

    Returns the duration-weighted complex grand average ``(F, n_eeg)``; the pooled
    null's RMS-over-sensors spectra ``(D, F)``; and the band max statistic
    ``(D, n_bands)`` -- for each draw and band, the largest band-mean
    magnitude-squared coherence over the sensors.
    """
    real = null = frequencies = None
    weight_sum = 0.0

    for path in tqdm(paths, desc="stimuli", unit="file"):
        with np.load(path) as saved:
            weight = float(saved["total_analysed_duration"])
            stimulus_real = saved["gammatone_coherence"].squeeze(1)        # (F, n_eeg)
            stimulus_null = saved["null_gammatone_coherence"].squeeze(2)   # (D, F, n_eeg)

            if real is None:
                frequencies = saved["frequencies"]
                real = np.zeros(stimulus_real.shape, np.complex128)
                null = np.zeros(stimulus_null.shape, np.complex64)

            # complex64 for the null: that is the precision the files are saved at, and a
            # float64 weight would silently promote every temporary to twice the size.
            real += stimulus_real * weight
            null += stimulus_null * np.float32(weight)
            weight_sum += weight

    real /= weight_sum
    null /= np.complex64(weight_sum)

    masks, _ = band_bin_masks(frequencies, bands)
    null_power = np.abs(null) ** 2                                  # (D, F, n_eeg)
    null_spectra = np.sqrt(null_power.mean(-1)).astype(np.float32)  # (D, F)
    band_null_max = np.stack(
        [null_power[:, mask, :].mean(1).max(1) for mask in masks], axis=1
    ).astype(np.float32)                                            # (D, n_bands)

    return frequencies, real.astype(np.complex64), null_spectra, band_null_max, weight_sum


def pooled_grand_average(reference):
    """The cached reduction for one reference, building it on first use.

    The cache carries the bands it was built over. ``band_null_max`` is one column per
    band, so a cache written for different bands would hand back thresholds belonging
    to the wrong frequency ranges -- with the right shape, and no error. Changing
    ``BASE_BANDS`` or ``EXTRA_BANDS`` therefore invalidates it rather than being
    silently ignored.
    """
    bands = bands_for(reference)
    cache = cache_dir(reference) / "grand_average_coherence.npz"
    if cache.exists():
        with np.load(cache) as saved:
            stale = "bands" not in saved or not np.array_equal(saved["bands"], bands)
            if not stale:
                return dict(
                    frequencies=saved["frequencies"], coherence=saved["coherence"],
                    null_spectra=saved["null_spectra"], band_null_max=saved["band_null_max"],
                    listening_seconds=float(saved["listening_seconds"]),
                    stimulus_seconds=dict(zip([str(s) for s in saved["stimuli"]],
                                              saved["stimulus_seconds"])),
                )
        print(f"  {reference}: cached bands differ from the current ones, re-pooling")

    paths = sorted(null_dir(reference).glob("*.npz"))
    frequencies, coherence, null_spectra, band_null_max, seconds = pool_null_files(paths, bands)
    stimulus_seconds = {}
    for path in paths:
        with np.load(path) as saved:
            stimulus_seconds[str(saved["name"])] = float(saved["total_analysed_duration"])

    np.savez(
        cache, frequencies=frequencies, coherence=coherence, null_spectra=null_spectra,
        band_null_max=band_null_max, listening_seconds=seconds, bands=bands,
        stimuli=list(stimulus_seconds), stimulus_seconds=list(stimulus_seconds.values()),
    )
    return dict(frequencies=frequencies, coherence=coherence, null_spectra=null_spectra,
                band_null_max=band_null_max, listening_seconds=seconds,
                stimulus_seconds=stimulus_seconds)


pooled = pooled_grand_average(PRIMARY_REFERENCE)
frequencies = pooled["frequencies"]
coherence = pooled["coherence"]
null_spectra = pooled["null_spectra"]
band_null_max = pooled["band_null_max"]
stimulus_seconds = pooled["stimulus_seconds"]

n_draws = null_spectra.shape[0]
print(f"{len(stimulus_seconds)} stimuli, {pooled['listening_seconds'] / 3600:.1f} "
      f"listening hours, {n_draws} pooled null draws, {coherence.shape[1]} channels")

## The spectrum and its null range

The trace is the RMS magnitude-squared coherence over the sensors, so one number per
frequency summarises the whole scalp. The shaded region is the full range of the 500
pooled null draws -- every draw falls inside it -- computed on that same statistic,
with their median drawn through it.

In [ ]:
spectrum = np.sqrt((np.abs(coherence) ** 2).mean(-1))        # (F,)

null_range = (null_spectra.min(0), null_spectra.max(0))
null_median = np.median(null_spectra, axis=0)

# Where the average leaves the null behind: the length of the leading run of bins all
# above every null draw, without interruption -- not simply the highest crossing, since
# isolated bins keep crossing well into the noise. Quoted, not drawn.
above_all = spectrum > null_range[1]
tail = above_all[1:].tolist()
leading_run = tail.index(False) if False in tail else len(tail)
crossover = frequencies[1:][leading_run - 1] if leading_run > 0 else None
crossover_text = (f"{crossover:.0f} Hz" if crossover is not None
                  else "no frequency (not above the null at the first analysed bin)")
print(f"grand average stays above all {n_draws} null draws out to {crossover_text} "
      f"(isolated bins cross up to {frequencies[1:][above_all[1:]].max():.0f} Hz)")
print(f"peak {spectrum[1:].max():.3f} at {frequencies[1:][np.argmax(spectrum[1:])]:.1f} Hz, "
      f"null median {np.median(null_median):.4f}")

## Band peaks

The vertical stems are placed where the coherence actually peaks in each band, not at
a frequency chosen by hand. Within a band the spectrum is a smooth hump on a falling
background, so a quartic is enough to locate its top without chasing bin-to-bin noise.

The fit is in log-frequency against log-coherence -- the space the spectrum is drawn
in, and the one where a band's shape is a hump rather than a steeply falling edge.
The band masks already drop every bin within 2 Hz of 50 Hz and its harmonics, so the
mains notch never pulls the fit down.

Peaks in the lowest two bands are reported on the 0.2 Hz grid the analysis runs on;
above that the fit is not that sharp, so they are snapped to whole Hz.

In [ ]:
POLY_ORDER = 4
PEAK_SAMPLES = 2001

# Draws the fits over the bins they were fitted to, as a check that each band's peak
# is a peak and not an artefact of the fit. A diagnostic, not one of the figures.
SHOW_FITS = True


def band_peak(values, mask, order=POLY_ORDER, samples=PEAK_SAMPLES, axis=None):
    """Fitted peak frequency inside one band, and whether it is a real peak.

    The maximum is read off a dense evaluation of the fit rather than from the roots
    of its derivative: same answer where there is an interior maximum, and it
    degrades gracefully where there is not. Landing on a band edge means the fit is
    monotonic across the band, i.e. that band carries no peak of its own, which the
    caller is told about rather than left to infer from a suspicious frequency.
    """
    # `axis` defaults to this notebook's own grid; the reference comparison passes
    # its own, so a peak is never fitted against the wrong frequency axis.
    log_frequency = np.log10((frequencies if axis is None else axis)[mask])
    coefficients = np.polyfit(log_frequency, np.log10(values[mask]), order)

    grid = np.linspace(log_frequency[0], log_frequency[-1], samples)
    index = int(np.argmax(np.polyval(coefficients, grid)))
    return float(10 ** grid[index]), coefficients, 0 < index < samples - 1


def snap(frequency, band_index):
    """Lowest two bands to the 0.2 Hz analysis grid, the rest to whole Hz."""
    return round(frequency / 0.2) * 0.2 if band_index < 2 else float(round(frequency))


band_masks_by_frequency, _ = band_bin_masks(frequencies)

BAND_CENTRES, band_fits, band_peak_is_interior, band_raw_peaks = [], [], [], []
for index, bins in enumerate(band_masks_by_frequency):
    peak, coefficients, interior = band_peak(spectrum, bins)
    BAND_CENTRES.append(snap(peak, index))
    band_fits.append(coefficients)
    band_peak_is_interior.append(interior)
    band_raw_peaks.append(peak)
BAND_CENTRES = np.array(BAND_CENTRES)

for (low, high), raw, centre, interior, bins in zip(
    BANDS, band_raw_peaks, BAND_CENTRES, band_peak_is_interior, band_masks_by_frequency,
):
    place = f"{centre:.1f}" if centre < 10 else f"{centre:.0f}"
    note = "" if interior else "   (no interior peak: fit is monotonic, pinned to a band edge)"
    print(f"{low:5.1f}-{high:5.1f} Hz: peak {place:>5s} Hz  "
          f"(fitted {raw:7.3f}, {int(bins.sum()):3d} bins){note}")


def plot_band_fits():
    """The fits over their own bins, one panel per band."""
    fig, axes = plt.subplots(1, len(BANDS), figsize=(2.6 * len(BANDS), 2.9),
                             tight_layout=True)
    for ax, (low, high), bins, centre, coefficients, interior in zip(
        axes, BANDS, band_masks_by_frequency, BAND_CENTRES, band_fits,
        band_peak_is_interior,
    ):
        # Every bin in the window, so the dropped mains bins show up as gaps in the
        # black points while the grey line still runs through them.
        window = (frequencies >= low) & (frequencies <= high)
        ax.plot(frequencies[window], spectrum[window], color=HAIRLINE, lw=0.8, zorder=1)
        ax.plot(frequencies[bins], spectrum[bins], color=INK, lw=0, marker=".", ms=2.2,
                zorder=2)

        grid = np.logspace(*np.log10(frequencies[bins][[0, -1]]), 400)
        ax.plot(grid, 10 ** np.polyval(coefficients, np.log10(grid)),
                color="#c8442e", lw=1.6, zorder=3)
        ax.axvline(centre, color="#c8442e", ls="dotted", lw=1.2, zorder=4)

        dropped = int(window.sum() - bins.sum())
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlim(low, high)
        # Three plain ticks. minorticks_off before setting them, not tick_params:
        # a log axis spanning less than a decade -- which every band here does --
        # labels its minor ticks too, and hiding the marks leaves the labels behind.
        ax.minorticks_off()
        ax.set_xticks([low, centre, high], [f"{v:g}" for v in (low, centre, high)],
                      fontsize=TICK_FONTSIZE)
        ax.set_xlabel("Frequency (Hz)")
        ax.set_title(f"{low:g}-{high:g} Hz   peak {centre:g}" + ("" if interior else "  (EDGE)")
                     + f"\n{dropped} mains bins dropped", fontsize=TICK_FONTSIZE)
        bare(ax)

    axes[0].set_ylabel("RMS coherence magnitude")
    axes[0].legend(
        [Line2D([], [], color=INK, lw=0, marker=".", ms=4),
         Line2D([], [], color=HAIRLINE, lw=1.2),
         Line2D([], [], color="#c8442e", lw=1.6)],
        ["fitted bins", "all bins", f"order-{POLY_ORDER} fit"],
        fontsize=SMALL_FONTSIZE, frameon=False, loc="lower left",
    )
    fig.suptitle(f"Band fits in log-log space -- reference: {PRIMARY_REFERENCE}",
                 fontsize=DEFAULT_FONTSIZE, y=1.04)
    return fig


if SHOW_FITS:
    plot_band_fits()
    plt.show()

## Band topographies

For each band the sensor-wise statistic is the mean magnitude-squared coherence over
that band's bins, mains harmonics excluded. A sensor is marked significant when it
exceeds the band's null threshold: the `1 - alpha/n_bands` quantile of the pooled
null's *maximum over sensors*, which controls the family-wise error across the scalp
within a band and is Bonferroni-corrected across the five bands.

In [ ]:
power = np.abs(coherence) ** 2                                # (F, n_eeg)
quantile = 100 * (1 - ALPHA / len(BANDS))                     # Bonferroni over the bands

band_topographies, band_masks, band_thresholds = [], [], []
for index, bins in enumerate(band_masks_by_frequency):
    observed = power[bins].mean(0)                            # (n_eeg,)
    threshold = np.percentile(band_null_max[:, index], quantile)
    band_topographies.append(np.sqrt(observed))
    band_masks.append(observed > threshold)
    band_thresholds.append(threshold)

for (low, high), mask, threshold in zip(BANDS, band_masks, band_thresholds):
    print(f"{low:5.1f}-{high:5.1f} Hz: {int(mask.sum()):2d}/{len(mask)} sensors "
          f"above {threshold:.2e}")

## Speech unit rates and pitch

Panel (b) puts the stimuli's own timescales on the same axis as the coherence. Each
unit's onsets become an instantaneous rate in 5 s windows stepped by 1 s, pooled over
stimuli by listening time; the pitch histogram is the same pooling over the F0
distributions, so both are read on the frequency axis above.

In [ ]:
UNIT_FILES = {
    "Int. units": "iu_onsets.json",
    "Words": "word_onsets.json",
    "Syllables": "nuclei_onsets_dejong.json",
    "Phonemes": "phoneme_onsets.json",
}
UNIT_COLOURS = ["#ef476f", "#ffd166", "#06d6a0", "#118ab2"]
PITCH_COLOUR = "#073b4c"

RATE_BINS = np.arange(-0.1, 512.2, 0.2)
RATE_CENTRES = (RATE_BINS[:-1] + RATE_BINS[1:]) / 2


def windowed_rates(onsets, window_len_s=5.0, step_s=1.0):
    """Instantaneous unit rate in overlapping windows. Matches figure 1's definition."""
    onsets = np.asarray(onsets, dtype=float)
    starts = np.arange(0, onsets[-1] - window_len_s, step_s)
    counts = np.array([
        np.sum((onsets >= start) & (onsets < start + window_len_s)) for start in starts
    ])
    return counts / window_len_s


def pooled_rate_histogram(filename):
    """Listening-time-weighted histogram of instantaneous rates over the analysed stimuli."""
    onsets_by_stimulus = json.loads((SPEECH_UNITS_DIR / filename).read_text())
    pooled_counts = np.zeros(len(RATE_CENTRES))
    for name, seconds in stimulus_seconds.items():
        counts, _ = np.histogram(windowed_rates(onsets_by_stimulus[name]), RATE_BINS)
        pooled_counts += counts * seconds
    pooled_counts[0] = 0.0  # windows with no unit in them are not a rate
    return pooled_counts


unit_rates = {label: pooled_rate_histogram(name) for label, name in UNIT_FILES.items()}

pitch_histogram, pitch_bins = None, None
for name, seconds in stimulus_seconds.items():
    with np.load(PITCH_DIR / f"{name}_pitch.npz") as pitch:
        smoothed = median_filter(pitch["counts"].astype(float), size=30) * seconds
        pitch_bins = pitch["bin_centres"]
    pitch_histogram = smoothed if pitch_histogram is None else pitch_histogram + smoothed

for label, histogram in unit_rates.items():
    print(f"{label:11s} modal rate {RATE_CENTRES[np.argmax(histogram)]:5.1f} Hz")
print(f"{'Pitch':11s} modal F0   {pitch_bins[np.argmax(pitch_histogram)]:5.1f} Hz")

## Figure 1 -- grand-average coherence

Each panel is a function of one axis, so the layout stays separate from the drawing.

In [ ]:
XLIM = (0.09, 750)   # the left margin carries the null-extent brackets
YLIM = (8e-4, 0.5)

# Where each topography sits, in axis coordinates, and how big it is.
INSET_Y = 0.68
INSET_MAX_SIZE = 0.25


def inset_layout(n_slots, scale=1.0):
    """Evenly spaced x positions for `n_slots` topographies, and their size.

    `scale` grows the topographies in one place rather than at the point of use, so
    the positions are laid out against the final size -- the last inset cannot run
    off the panel -- and the leader lines, which anchor on the inset's centre, point
    at where the topography actually is.

    The insets are square in axes fractions but the panel is far wider than it is
    tall, so a topography is limited by the box's height; the extra width is slack,
    not overlap.
    """
    size = min(INSET_MAX_SIZE, 1.3 / n_slots) * scale
    return np.linspace(0.01, 1.0 - size, n_slots), size


# The single-reference figure spaces its own bands evenly across the panel.
INSET_X, INSET_SIZE = inset_layout(len(BANDS))

STEM_LABEL_Y = YLIM[0] * 1.1   # the row the stem labels sit on
STEM_LINE_POINTS = 8.0         # one line, for a label lifted clear of its neighbour


def stem_label_width(ax, label, fontsize=SMALL_FONTSIZE):
    """How wide a stem label renders, in decades of this axis.

    Measured rather than assumed: whether two labels collide depends on the font
    size, the figure width and the axis limits together, so a gap fixed in decades
    silently stops being right as soon as any of them changes.
    """
    probe = ax.text(XLIM[0], YLIM[0], label, fontsize=fontsize)
    box = probe.get_window_extent(ax.figure.canvas.get_renderer())
    left, right = ax.transData.inverted().transform([(box.x0, 0), (box.x1, 0)])[:, 0]
    probe.remove()
    return float(np.log10(right / left))


def stem_labels(ax, centres):
    """Label text for each band peak, and which of them must be lifted a line.

    Peaks are not evenly spread, so a label can reach into the next one. Where that
    happens the earlier of the pair goes up a line and the later stays on the row,
    which keeps every label beside its own stem.
    """
    labels = [f"{centre:.1f} Hz" if i < 2 else f"{int(centre)} Hz"
              for i, centre in enumerate(centres)]
    widths = [stem_label_width(ax, label) for label in labels]
    lifted = [
        i < len(labels) - 1 and np.log10(centres[i + 1] / centre) < widths[i]
        for i, centre in enumerate(centres)
    ]
    return labels, lifted


def plot_coherence_spectrum(ax, significance_markers=True):
    """Grand-average coherence over the range of the pooled null, with band topographies.

    `significance_markers=False` draws the topographies without the dots marking the
    sensors that clear the max-statistic threshold.
    """
    # The full extent of the null, as one region: with nothing layered over it the
    # fill carries on its own, so it is darker than it was under the inner band.
    ax.fill_between(frequencies[1:], *[bound[1:] for bound in null_range],
                    color=NULL_FILL, alpha=0.85, lw=0.0, zorder=1)
    #ax.plot(frequencies[1:], null_median[1:], color=MUTED, lw=0.8, zorder=3)
    ax.plot(frequencies[1:], spectrum[1:], color=INK, lw=0.8, zorder=0)

    # Lettered in place rather than in a legend: both labels are wrapped to two lines so
    # they stay clear of the 0.6 Hz stem.
    ax.text(0.105, 0.021, "EEG-envelope\ncoherence", color=INK,
            fontsize=TICK_FONTSIZE, va="top", linespacing=0.9)
    ax.text(0.105, 0.0016, "Noise\nlevel", color=MUTED,
            fontsize=TICK_FONTSIZE, va="bottom", linespacing=0.9)

    ax.set_ylabel("RMS coherence magnitude")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(*XLIM)
    ax.set_ylim(*YLIM)
    ax.set_xticks([0.1, 1, 10, 100], [0.1, 1, 10, 100], fontsize=TICK_FONTSIZE)
    ax.set_yticks([0.001, 0.01, 0.1], ["$10^{-3}$", "$10^{-2}$", "$10^{-1}$"],
                  fontsize=TICK_FONTSIZE)
    bare(ax)

    stem_label_text, stem_label_lifted = stem_labels(ax, BAND_CENTRES)
    for i, (band, centre) in enumerate(zip(BANDS, BAND_CENTRES)):
        axins = ax.inset_axes([INSET_X[i], INSET_Y, INSET_SIZE, INSET_SIZE],
                              transform=ax.transAxes)
        plot_topomap(
            band_topographies[i], INFO, cmap="Reds", contours=0,
            mask=band_masks[i] if significance_markers else None, sensors=False,
            mask_params=dict(marker="o", markerfacecolor="k", markeredgecolor="none",
                             markersize=1.5),
            show=False, axes=axins,
        )
        low, high = band
        low_label = f"{low:.1f}" if low < 1 else f"{int(low)}"
        axins.set_title(f"{low_label}-{int(high)} Hz", fontsize=TICK_FONTSIZE)

        # A dotted stem from the centre frequency up to the trace, then on to the inset.
        elbow = float(spectrum[np.argmin(np.abs(frequencies - centre))])
        ax.vlines(centre, YLIM[0], elbow, color=INK, lw=0.8, linestyles="dotted", zorder=5)
        # Offset in points, not a factor on a log axis: one line is one line
        # wherever on the axis the label happens to sit.
        ax.annotate(stem_label_text[i], xy=(centre * 1.04, STEM_LABEL_Y),
                    xycoords="data", textcoords="offset points",
                    xytext=(0, STEM_LINE_POINTS if stem_label_lifted[i] else 0),
                    color=INK, fontsize=SMALL_FONTSIZE, ha="left", va="baseline")

        # The inset's anchor is in axis coordinates; the trace is in data coordinates.
        anchor = ax.transData.inverted().transform(
            ax.transAxes.transform((INSET_X[i] + INSET_SIZE / 2, INSET_Y))
        )
        ax.plot([centre, anchor[0]], [elbow, anchor[1]],
                color=INK, lw=0.8, ls="dotted", zorder=5)

    return ax


def plot_unit_rates(ax):
    """Rates of speech units in the stimuli, and their pitch, on the same axis."""
    for (label, histogram), colour in zip(unit_rates.items(), UNIT_COLOURS):
        normalised = histogram / histogram.max()
        ax.step(RATE_CENTRES, normalised, c=colour, label=label, where="mid", lw=1.0)
        ax.fill_between(RATE_CENTRES, normalised, alpha=0.4, color=colour, step="mid", lw=0)

    audible = pitch_bins > 60
    normalised = pitch_histogram[audible] / pitch_histogram.max()
    ax.step(pitch_bins[audible], normalised, c=PITCH_COLOUR, label="Pitch", where="mid", lw=1.0)
    ax.fill_between(pitch_bins[audible], normalised, alpha=0.4, color=PITCH_COLOUR,
                    step="mid", lw=0)

    ax.set_xscale("log")
    ax.set_xlim(*XLIM)
    ax.set_ylim(0, 1.08)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Normalized count", labelpad=11)
    ax.set_xticks([0.1, 1, 10, 100], [0.1, 1, 10, 100], fontsize=TICK_FONTSIZE)
    ax.set_yticks([0, 0.5, 1.0], ["0", "0.5", "1"], fontsize=TICK_FONTSIZE)
    ax.legend(loc=(0.565, 0.055), frameon=False, fontsize=TICK_FONTSIZE,
              handlelength=1.2, labelspacing=0.25)
    return bare(ax)

In [ ]:
fig = plt.figure(figsize=FIGSIZE)
gs = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[10, 4], hspace=0.3)

ax_spectrum = fig.add_subplot(gs[0])
ax_rates = fig.add_subplot(gs[1], sharex=ax_spectrum)
ax_spectrum.tick_params(axis="x", labelbottom=False)
ax_rates.tick_params(axis="x", labelbottom=True)

plot_coherence_spectrum(ax_spectrum)
plot_unit_rates(ax_rates)

ax_spectrum.set_title("EEG-envelope coherence across the spectrum")
ax_rates.set_title("Rates of speech units in the stimuli")

for ax, label, y in [(ax_spectrum, "a)", 1.06), (ax_rates, "b)", 1.10)]:
    ax.text(-0.075, y, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

# Named for the reference, so no two references can overwrite each other's output.
stem = f"grand_average_coherence_{PRIMARY_REFERENCE}"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

## Figure 2 -- the same spectrum under all three references

One row per EEG reference, on a shared frequency axis, so the reference's effect on
the grand average can be read off directly rather than by flipping between figures.
Each row is panel (a) of figure 1 for that reference: the duration-weighted grand
average, the full range of its own 500 pooled null draws, and the five band
topographies with their significant sensors marked.

Two things differ between rows and are worth watching. The **channel count** is not
the same -- FCz loses the channel it is referenced to, and its topography is
interpolated over that hole -- and the **band peaks are refitted per reference**, so
the stems need not line up. Where they do, the peak is a property of the signal
rather than of the montage.

`PRIMARY_REFERENCE` does not apply here; this figure always loads all three.

In [ ]:
REFERENCE_LABELS = {
    "avg": "Average reference",
    "linked_mastoids": "Linked mastoids",
    "FCz": "FCz reference",
}

# Row titles. The channel count is filled from the data rather than written in, so it
# cannot drift from what the files actually carry.
REFERENCE_TITLES = {
    "avg": "Average reference ({n} channels)",
    "linked_mastoids": "Linked-mastoids reference ({n} channels)",
    "FCz": "FCz reference ({n} channels; FCz was dropped)",
}


def reference_summary(reference):
    """Everything one row of the comparison needs, for one reference.

    The same reductions the single-reference cells above perform, gathered into one
    call so the three rows can be built in a loop. Deliberately not a refactor of
    those cells: they are what figure 1 runs on, and this figure should not be able
    to change figure 1 by changing them.
    """
    bands = bands_for(reference)
    pooled = pooled_grand_average(reference)
    axis = pooled["frequencies"]
    power = np.abs(pooled["coherence"]) ** 2                     # (F, n_eeg)
    trace = np.sqrt(power.mean(-1))                              # (F,)

    masks, _ = band_bin_masks(axis, bands)
    threshold_quantile = 100 * (1 - ALPHA / len(bands))          # Bonferroni over bands

    centres, topographies, significant = [], [], []
    for index, bins in enumerate(masks):
        observed = power[bins].mean(0)
        topographies.append(np.sqrt(observed))
        significant.append(
            observed > np.percentile(pooled["band_null_max"][:, index], threshold_quantile)
        )
        centres.append(snap(band_peak(trace, bins, axis=axis)[0], index))

    null = pooled["null_spectra"]
    return dict(
        reference=reference, label=REFERENCE_LABELS[reference],
        title=REFERENCE_TITLES[reference], bands=bands,
        frequencies=axis, spectrum=trace,
        null_range=(null.min(0), null.max(0)), null_median=np.median(null, axis=0),
        centres=np.array(centres), topographies=topographies, significant=significant,
        info=info_for(reference), n_channels=pooled["coherence"].shape[1],
        listening_hours=pooled["listening_seconds"] / 3600.0,
    )


summaries = [reference_summary(reference) for reference in REFERENCES]

# One slot per distinct band over all references, in frequency order. A row places each
# of its topographies in its band's slot and leaves the others empty, so a band occupies
# the same horizontal position in every panel and the rows can be read down a column.
COMPARISON_BANDS = sorted({tuple(band) for s in summaries for band in s["bands"]})
COMPARISON_INSET_SCALE = 1.25   # the rows have height to spare; figure 1 does not
COMPARISON_X, COMPARISON_SIZE = inset_layout(len(COMPARISON_BANDS),
                                             scale=COMPARISON_INSET_SCALE)

for summary in summaries:
    peaks = "  ".join(f"{low:g}-{high:g}:{c:g}"
                      for (low, high), c in zip(summary["bands"], summary["centres"]))
    print(f"{summary['label']:18s} {summary['n_channels']:2d} ch  "
          f"{summary['listening_hours']:6.1f} h   band:peak (Hz)  {peaks}")
print(f"\n{len(COMPARISON_BANDS)} slots across the comparison: "
      + ", ".join(f"{low:g}-{high:g}" for low, high in COMPARISON_BANDS))

In [ ]:
ROW_TITLE_PAD = 4   # the topographies sit below the axes top, so the title can come down

# The comparison's topographies are larger than figure 1's and every row labels its
# own bands, so they sit lower in the panel: that keeps the band labels inside the
# axes and leaves the row title free to come right down to the top of the frame.
# Raise or lower the row's topographies here, not at the `inset_axes` call: the
# leader lines anchor on this same value, so scaling it at the point of use
# moves the topography and leaves the line pointing where it used to be.
COMPARISON_INSET_Y = 0.65


def plot_reference_row(ax, summary, show_band_titles=False, show_xlabel=False):
    """One reference's grand average, with its null range and band topographies.

    Every row carries its own tick labels: the three are read against each other, and
    a row whose axis is three panels away is harder to place than one repeated scale.
    The axis *label* stays on the bottom row only, where it is not repetition.
    """
    axis, trace = summary["frequencies"], summary["spectrum"]

    ax.fill_between(axis[1:], *[bound[1:] for bound in summary["null_range"]],
                    color=NULL_FILL, alpha=0.6, lw=0, zorder=1)
    #ax.plot(axis[1:], summary["null_median"][1:], color=MUTED, lw=0.8, zorder=3)
    ax.plot(axis[1:], trace[1:], color=INK, lw=0.8, zorder=0)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(*XLIM)
    ax.set_ylim(*YLIM)
    ax.set_xticks([0.1, 1, 10, 100], [0.1, 1, 10, 100], fontsize=TICK_FONTSIZE)
    ax.set_yticks([0.001, 0.01, 0.1], ["$10^{-3}$", "$10^{-2}$", "$10^{-1}$"],
                  fontsize=TICK_FONTSIZE)
    ax.set_ylabel("RMS coherence\nmagnitude", fontsize=TICK_FONTSIZE)
    ax.tick_params(axis="x", labelbottom=True)
    if show_xlabel:
        ax.set_xlabel("Frequency (Hz)")
    bare(ax)

    ax.set_title(summary["title"].format(n=summary["n_channels"]),
                 fontsize=DEFAULT_FONTSIZE, pad=ROW_TITLE_PAD)

    # Named in place rather than in a legend, as in figure 1: two lines each, so they
    # stay clear of the lowest stem.
    ax.text(0.105, 0.021, "EEG-envelope\ncoherence", color=INK,
            fontsize=TICK_FONTSIZE, va="top", linespacing=0.9)
    ax.text(0.105, 0.0016, "Noise\nlevel", color=MUTED,
            fontsize=TICK_FONTSIZE, va="bottom", linespacing=0.9)

    stem_label_text, stem_label_lifted = stem_labels(ax, summary["centres"])
    for i, (band, centre) in enumerate(zip(summary["bands"], summary["centres"])):
        # The slot this band occupies in every panel, not this row's i'th position:
        # linked mastoids carries a band the others do not, and without this its
        # topographies would all shift relative to theirs.
        slot = COMPARISON_BANDS.index(tuple(band))
        axins = ax.inset_axes([COMPARISON_X[slot], COMPARISON_INSET_Y, COMPARISON_SIZE,
                               COMPARISON_SIZE], transform=ax.transAxes)
        plot_topomap(
            summary["topographies"][i], summary["info"], cmap="Reds", contours=0,
            mask=summary["significant"][i], sensors=False,
            mask_params=dict(marker="o", markerfacecolor="k", markeredgecolor="none",
                             markersize=1.2),
            show=False, axes=axins,
        )
        low, high = band
        low_label = f"{low:.1f}" if low < 1 else f"{int(low)}"
        axins.set_title(f"{low_label}-{int(high)} Hz", fontsize=SMALL_FONTSIZE, pad=-4)

        elbow = float(trace[np.argmin(np.abs(axis - centre))])
        ax.vlines(centre, YLIM[0], elbow, color=INK, lw=0.8, linestyles="dotted", zorder=5)

        ax.annotate(stem_label_text[i], xy=(centre * 1.04, STEM_LABEL_Y),
                    xycoords="data", textcoords="offset points",
                    xytext=(0, STEM_LINE_POINTS*0.8 if stem_label_lifted[i] else 0),
                    color=INK, fontsize=SMALL_FONTSIZE, ha="left", va="baseline")

        anchor = ax.transData.inverted().transform(
            ax.transAxes.transform(
                (COMPARISON_X[slot] + COMPARISON_SIZE / 2, COMPARISON_INSET_Y))
        )
        ax.plot([centre, anchor[0]], [elbow, anchor[1]],
                color=INK, lw=0.8, ls="dotted", zorder=5)

    return ax

In [ ]:
# The figure's height and the gap between rows. The gap must still hold a title and a
# tick-labelled axis on every row, not just the bottom one; 0.28 leaves ~0.45 in for that.
REFERENCE_FIGSIZE = (FIGSIZE[0], 7.4)
REFERENCE_HSPACE = 0.28

fig = plt.figure(figsize=REFERENCE_FIGSIZE)
gs = gridspec.GridSpec(len(summaries), 1, figure=fig, hspace=REFERENCE_HSPACE)

# A band is titled on the first row that carries it: the top row has no 20-28 Hz
# topography, so that slot would otherwise go unlabelled.
titled = set()
axes = []
for row, summary in enumerate(summaries):
    ax = fig.add_subplot(gs[row], sharex=axes[0] if axes else None)
    new_bands = {tuple(b) for b in summary["bands"]} - titled
    plot_reference_row(ax, summary, show_band_titles=new_bands,
                       show_xlabel=row == len(summaries) - 1)
    titled |= new_bands
    axes.append(ax)

# Level with each row's title, so a letter reads with its own panel rather than hanging in
# the gap under the row above.
REFERENCE_LABEL_Y = 1.01
for ax, label in zip(axes, ["a)", "b)", "c)"]):
    ax.text(-0.09, REFERENCE_LABEL_Y, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

fig.savefig(OUT_DIR / "reference_comparison.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / "reference_comparison.png", bbox_inches="tight", pad_inches=0.3,
            dpi=200)
plt.show()

## Figure 6 -- the grand average resolved over channels

Figure 1's panel (a) collapses the scalp into one RMS trace. This section opens it
back up: the same pooled grand average per channel, with the sensors partitioned into
five scalp regions.

Four views of one array. The heatmap is every channel against frequency, blocked by
region; beside it, each region's own RMS trace over its own null; below, every
channel's trace coloured by where it sits on the head; and a sensor map keying the
regions.

The quantity is `abs(coherence)` -- the magnitude of the *pooled complex* grand
average, the same array figure 1 and the band topographies are built from. Averaging
magnitudes across trials instead would not let noise cancel and sits higher; the two
are not interchangeable.

Regions are intersected with the channels the reference actually has, so the FCz
reference simply drops FCz from the central group rather than failing.

In [ ]:
import mne
from matplotlib.colors import ListedColormap, LogNorm
from matplotlib.ticker import LogLocator, NullFormatter
# Private MNE helpers: they draw the head outline and the position-to-colour map that
# `plot_sensors` uses internally, and there is no public route to either.
from mne.viz.evoked import _rgb
from mne.viz.topomap import _draw_outlines, _get_pos_outlines

# The five regions partition the scalp, in the order they are stacked.
BASE_CHANNEL_GROUPS = {
    "Frontal": ["AF7", "F7", "F5", "F6", "F8", "AF8", "Fp1", "Fpz", "Fp2",
                "F3", "F4", "AF3", "AF4", "AFz"],
    "Central": ["F1", "Fz", "F2", "FC1", "FCz", "FC2", "C1", "Cz", "C2",
                "CP1", "CPz", "CP2", "P1", "Pz", "P2"],
    "Occipital": ["PO7", "PO3", "POz", "PO4", "PO8", "O1", "Oz", "O2", "Iz"],
    "Right Temporal": ["P10", "P8", "P6", "P4", "CP4", "CP6", "TP8",
                       "C4", "C6", "T8", "FC4", "FC6", "FT8"],
    "Left Temporal": ["P9", "P7", "P5", "P3", "CP3", "CP5", "TP7",
                      "C3", "C5", "T7", "FC3", "FC5", "FT7"],
}


##['TP7', 'P7', 'P9', 'PO7', 'O1', 'Iz', 'Oz', 'T8', 'TP8', 'P8', 'P10', 'PO8', 'O2'] 'P10',
#['Fp1', 'AF3', 'F1', 'F3', 'FC3', 'FC1', 'C1', 'C3', 'Fpz', 'Fp2', 'AF4', 'AFz', 'Fz', 'F2', 'F4', 'FC4', 'FC2', 'FCz', 'Cz', 'C2', 'C4', 'C6'] C6§
#['CP5', 'CP3', 'CP1', 'P1', 'P3', 'P5', 'PO3', 'POz', 'Pz', 'CPz', 'CP6', 'CP4', 'CP2', 'P2', 'P4', 'P6', 'PO4']
#['AF7', 'F5', 'F7', 'FT7', 'FC5', 'C5', 'T7', 'AF8', 'F6', 'F8', 'FT8', 'FC6']
# BASE_CHANNEL_GROUPS = {
#     "Fronto-central": ['Fp1', 'AF3', 'F1', 'F3', 'FC3', 'FC1', 'C1', 'C3', 'Fpz', 'Fp2', 'AF4', 'AFz', 'Fz', 'F2', 'F4', 'FC4', 'FC2', 'FCz', 'Cz', 'C2', 'C4'],
#     "Occipital": ['CP5', 'CP3', 'CP1', 'P1', 'P3', 'P5', 'PO3', 'POz', 'Pz', 'CPz', 'CP6', 'CP4', 'CP2', 'P2', 'P4', 'P6', 'PO4', 'P10'],
#     "Anterior": ['TP7', 'P7', 'P9', 'PO7', 'O1', 'Iz', 'Oz', 'TP8', 'P8',  'PO8', 'O2'],
#     "Left Temporal": ['AF7', 'F5', 'F7', 'FT7', 'FC5', 'C5', 'T7'],
#     "Right Temporal": ['AF8', 'F6', 'F8', 'FT8', 'FC6', 'C6', 'T8']
# }
GROUP_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
HEATMAP_XLIM = (0, 300)


def channel_groups_for(info):
    """The regions, restricted to the channels this reference carries.

    FCz is in the central group and is gone under its own reference, so the groups are
    intersected with the info rather than assumed. The assertion is the point: every
    channel must land in exactly one region, or the heatmap silently omits sensors.
    """
    groups = {name: [ch for ch in group if ch in info.ch_names]
              for name, group in BASE_CHANNEL_GROUPS.items()}
    assigned = [ch for group in groups.values() for ch in group]
    unassigned = [ch for ch in info.ch_names if ch not in assigned]
    duplicated = [ch for ch in assigned if assigned.count(ch) > 1]
    assert not unassigned and not duplicated, (unassigned, duplicated)
    return groups


CHANNEL_GROUPS = channel_groups_for(INFO)
GROUP_ORDER = [ch for group in CHANNEL_GROUPS.values() for ch in group]
CHANNEL_INDEX = [INFO.ch_names.index(ch) for ch in GROUP_ORDER]
GROUP_SIZES = [len(group) for group in CHANNEL_GROUPS.values()]

channel_coherence = np.abs(coherence)                        # (F, n_eeg)
grouped_coherence = channel_coherence[:, CHANNEL_INDEX].T    # (n_eeg, F), in group order

for name, size in zip(CHANNEL_GROUPS, GROUP_SIZES):
    print(f"{name:16s} {size:2d} sensors")
print(f"{'total':16s} {sum(GROUP_SIZES):2d} of {len(INFO.ch_names)} channels")

In [ ]:
# Figure 1's cache keeps the null's RMS over *all* sensors, which would draw the same
# line under every region. This is the same pooled null -- identical draws, weighted
# identically -- reduced over each region's own sensors instead. It needs the full
# (D, F, n_eeg) null, which the other cache does not keep, so it costs one more pass
# over the per-stimulus files and is cached separately.


def pool_group_null(paths, info, groups):
    """Per-region RMS null spectra, ``(n_draws, F, n_groups)``."""
    null, weight_sum = None, 0.0
    for path in tqdm(paths, desc="stimuli", unit="file"):
        with np.load(path) as saved:
            weight = np.float32(saved["total_analysed_duration"])
            stimulus_null = saved["null_gammatone_coherence"].squeeze(2)   # (D, F, n_eeg)
            if null is None:
                null = np.zeros(stimulus_null.shape, np.complex64)
            null += stimulus_null * weight
            weight_sum += float(weight)

    null /= np.complex64(weight_sum)
    power = np.abs(null) ** 2
    return np.stack([
        np.sqrt(power[:, :, [info.ch_names.index(ch) for ch in channels]].mean(-1))
        for channels in groups.values()
    ], axis=-1).astype(np.float32)


def group_null_spectra(reference, info, groups):
    """The cached per-region null for one reference, building it on first use.

    The cache carries the region names it was built for: the array's last axis is one
    column per region, so a cache built for a different partition would line up by
    shape and be wrong by content.
    """
    cache = cache_dir(reference) / "channel_group_null.npz"
    if cache.exists():
        with np.load(cache) as saved:
            if [str(name) for name in saved["groups"]] == list(groups):
                return saved["group_null_rms"]
        print(f"  {reference}: cached regions differ from the current ones, re-pooling")

    group_null_rms = pool_group_null(sorted(null_dir(reference).glob("*.npz")), info, groups)
    np.savez(cache, group_null_rms=group_null_rms, groups=list(groups))
    return group_null_rms


group_null_rms = group_null_spectra(PRIMARY_REFERENCE, INFO, CHANNEL_GROUPS)
group_null_median = np.median(group_null_rms, axis=0)            # (F, n_groups)
group_null_inner = np.percentile(group_null_rms, [1, 99], axis=0)

# Pooling the regions back together must return the all-sensor null figure 1 uses.
# Weighted by region size, since an RMS averages in power.
recombined = np.sqrt(np.average(group_null_rms ** 2, axis=-1, weights=GROUP_SIZES))
difference = np.abs(np.median(recombined, 0) / null_median - 1).max()
print(f"per-region null vs all-sensor null, max relative difference: {difference:.2e}")
assert difference < 1e-4, "the per-region null does not recombine to the pooled one"
for i, name in enumerate(CHANNEL_GROUPS):
    print(f"  {name:16s} median noise level {np.median(group_null_median[1:, i]):.5f}")

In [ ]:
# Same 6.27 in text width as the other figures, so all of them carry text at the same
# physical size in the document.
CHANNELS_FIGSIZE = (6.27, 7.0)

fig = plt.figure(figsize=CHANNELS_FIGSIZE)
outer = fig.add_gridspec(2, 2, height_ratios=[1.5, 1], width_ratios=[1.2, 1],
                         hspace=0.34, wspace=0.44)
# Column ratios are 1.2:1, not 2.4:1 -- at this width the RMS stack was squeezed into
# the right margin. The gutter must hold the colourbar's tick labels and the stack's
# together, and wspace is a fraction of the *average* column width, so raising it alone
# would take the space off both columns equally.

# --- a) every channel against frequency, blocked by region ----------------------
ax_heat = fig.add_subplot(outer[0, 0])
mesh = ax_heat.pcolormesh(
    frequencies, np.arange(len(CHANNEL_INDEX)), grouped_coherence, shading="auto",
    cmap="viridis",
    # Floor at the 1st percentile, not the minimum: a few bins where the complex
    # average cancels would otherwise stretch the scale two decades too far.
    norm=LogNorm(vmin=np.percentile(grouped_coherence, 1), vmax=grouped_coherence.max()),
    rasterized=True,   # 2561x64 cells as vector art is ~4 MB of PDF for no visible gain
)
ax_heat.set_xlim(*HEATMAP_XLIM)

ticks, labels, offset = [], [], 0
for name, channels in CHANNEL_GROUPS.items():
    ticks.append(offset + (len(channels) - 1) / 2)
    labels.append(name)
    offset += len(channels)
    if offset < len(CHANNEL_INDEX):
        ax_heat.axhline(offset - 0.5, color="white", lw=0.8, alpha=0.6)

ax_heat.set_yticks(ticks)
# Angled and wrapped: upright on one line these eat most of the panel's left margin.
ax_heat.set_yticklabels([label.replace(" Temporal", "\nTemporal") for label in labels],
                        fontsize=TICK_FONTSIZE, rotation=40, ha="right",
                        va="center", rotation_mode="anchor")
ax_heat.invert_yaxis()          # frontal at the top, as in the group order
ax_heat.set_xlabel("Frequency (Hz)")
ax_heat.set_title("Log-magnitude coherence\nby channel")
colourbar = fig.colorbar(mesh, cax=ax_heat.inset_axes([1.015, 0.0, 0.025, 1.0]))
# No caption on the bar: rotated beside it, it lands on the RMS panels' y-label; above
# it, on this panel's title. The title already names the quantity.
colourbar.ax.tick_params(labelsize=TICK_FONTSIZE)
colourbar.outline.set_visible(False)

# --- b) each region's RMS trace over its own null --------------------------------
# Equal heights and one shared y-extent, so the regions are compared against each other
# rather than against the heatmap's block sizes. Deliberately NOT sharex with the
# heatmap: these are log-frequency, and sharex would drag its linear axis along.
group_rms, offset = {}, 0
for name, channels in CHANNEL_GROUPS.items():
    block = grouped_coherence[offset:offset + len(channels), :]
    group_rms[name] = np.sqrt(np.mean(block ** 2, axis=0))
    offset += len(channels)

stacked = np.array(list(group_rms.values()))[:, 1:]
# Top pinned above 1e-1 so the axis carries three whole decades of ticks.
rms_ylim = (stacked.min() * 0.8, max(stacked.max() * 1.3, 0.115))

gs_rms = outer[0, 1].subgridspec(len(CHANNEL_GROUPS), 1, hspace=0.15)
group_axes = []
for i, (name, rms) in enumerate(group_rms.items()):
    ax_rms = fig.add_subplot(gs_rms[i, 0],
                             sharex=group_axes[0] if group_axes else None,
                             sharey=group_axes[0] if group_axes else None)
    ax_rms.plot(frequencies[1:], rms[1:], lw=0.8, color=GROUP_COLOURS[i])
    # This region's own null, over exactly the sensors plotted above it. Drawn *over*
    # the trace, not under: where the trace vanishes into the grey it is inside the
    # null's central 98%, which is the thing worth seeing.
    ax_rms.fill_between(frequencies[1:], group_null_inner[0, 1:, i],
                        group_null_inner[1, 1:, i],
                        color=NULL_FILL, alpha=0.75, lw=0, zorder=3)
    ax_rms.plot(frequencies[1:], group_null_median[1:, i], color=MUTED, lw=0.7, zorder=4)
    ax_rms.text(0.97, 0.88, name, transform=ax_rms.transAxes, ha="right", va="top",
                fontsize=SMALL_FONTSIZE, color=GROUP_COLOURS[i])
    ax_rms.tick_params(labelbottom=False, labelsize=TICK_FONTSIZE)
    ax_rms.set_xscale("log")
    ax_rms.set_yscale("log")
    ax_rms.set_xlim(*XLIM)
    ax_rms.set_ylim(*rms_ylim)
    # Locators, not set_yticks: that replaces the locator outright and drops the log
    # minor ticks with it. numticks forces every decade to be labelled -- left alone
    # the locator strides over one in a panel this short.
    ax_rms.yaxis.set_major_locator(LogLocator(base=10.0, subs=(1.0,), numticks=12))
    ax_rms.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10),
                                              numticks=20))
    ax_rms.yaxis.set_minor_formatter(NullFormatter())
    bare(ax_rms)
    group_axes.append(ax_rms)

group_axes[-1].tick_params(labelbottom=True)
group_axes[-1].set_xticks([0.1, 1, 10, 100], [0.1, 1, 10, 100])   # as in figure 1
group_axes[-1].set_xlabel("Frequency (Hz)")
group_axes[0].set_title("RMS coherence by region", pad=8)

# --- c) every channel's trace, coloured by where it sits on the head -------------
ax_line = fig.add_subplot(outer[1, 0], sharex=ax_heat)
positions = np.array([channel["loc"][:3] for channel in INFO["chs"]])
channel_colours = _rgb(*positions.T)
for i in range(channel_coherence.shape[1]):
    # 64 alpha-blended traces over 2561 points each: rasterised for the same reason.
    ax_line.plot(frequencies[1:], channel_coherence[1:, i],
                 color=channel_colours[i], lw=0.4, alpha=0.25, rasterized=True)
ax_line.set_xlabel("Frequency (Hz)")
ax_line.set_ylabel("Coherence magnitude")
ax_line.set_title("Per-channel coherence profiles")
bare(ax_line)

ax_line_topo = ax_line.inset_axes([0.62, 0.55, 0.36, 0.36])
sensor_positions, outlines = _get_pos_outlines(INFO, np.arange(len(CHANNEL_INDEX)),
                                               sphere=(0, 0, 0, 0.095))
_draw_outlines(ax_line_topo, outlines)
ax_line_topo.scatter(*sensor_positions.T, c=channel_colours, s=4)
ax_line_topo.set_aspect("equal")
ax_line_topo.set_axis_off()

# --- d) the sensor map that keys the regions ------------------------------------
ax_legend = fig.add_subplot(outer[1, 1])
group_channel_index, offset = [], 0
for channels in CHANNEL_GROUPS.values():
    group_channel_index.append(CHANNEL_INDEX[offset:offset + len(channels)])
    offset += len(channels)

mne.viz.plot_sensors(
    INFO, kind="topomap", ch_groups=group_channel_index, axes=ax_legend,
    show=False, show_names=False, pointsize=12, linewidth=0,
    cmap=ListedColormap(GROUP_COLOURS), title=None,
)
ax_legend.set_title("Channel groups")
# Zoom out a little so the head does not fill the cell: with the legend tucked under
# it, the pair needs less of the page's bottom margin.
for get_limits, set_limits in ((ax_legend.get_xlim, ax_legend.set_xlim),
                               (ax_legend.get_ylim, ax_legend.set_ylim)):
    low, high = get_limits()
    centre, half_span = (low + high) / 2, (high - low) / 2 * 1.18
    set_limits(centre - half_span, centre + half_span)

ax_legend.legend(
    handles=[Line2D([0], [0], marker="o", color="none", markerfacecolor=colour,
                    markersize=5, label=name)
             for name, colour in zip(CHANNEL_GROUPS, GROUP_COLOURS)],
    loc="lower center", bbox_to_anchor=(0.5, -0.30), ncol=2, frameon=False,
    fontsize=TICK_FONTSIZE, handlelength=1.0, handletextpad=0.4, columnspacing=1.0,
)

fig.tight_layout()

# The colourbar narrows ax_heat's box to make room for itself, but ax_line sits in a
# separate gridspec cell that never gets the same narrowing -- sharex syncs the data
# range, not the width of the box. Match ax_line's box to ax_heat's final one.
heat_box, line_box = ax_heat.get_position(), ax_line.get_position()
ax_line.set_position([heat_box.x0, line_box.y0, heat_box.width, line_box.height])

# plot_sensors puts an equal aspect on ax_legend, which shrinks and re-centres its box
# inside the gridspec cell, so its title sits lower than (c)'s even though they share a
# row. Pin it to where (c)'s title lands, in figure coordinates.
title_pad_fig = (plt.rcParams["axes.titlepad"] / 72) / CHANNELS_FIGSIZE[1]
title_y_fig = ax_line.get_position().y1 + title_pad_fig
legend_box = ax_legend.get_position()
ax_legend.set_title("Legend of channel regions", pad=-0.2,
                    y=(title_y_fig - legend_box.y0) / legend_box.height)

# Panel markers last, once every box is final. The y is settled per *row* in figure
# coordinates and converted back into each axes' own fraction, so a) sits level with b)
# and c) with d) -- in axes coordinates each would follow its own top edge, and the two
# columns do not share one. The x offsets stay per-axes: the four boxes have very
# different left margins.
for row, labels_and_offsets, lift in [
    ([ax_heat, group_axes[0]], [("a)", -0.13), ("b)", -0.20)], 0.014),
    # The bottom row sits a little higher above its axes: (a)'s title is two lines and
    # (b)'s one, so the top row's markers already clear more.
    ([ax_line, ax_legend], [("c)", -0.13), ("d)", -0.14)], 0.030),
]:
    row_top = max(ax.get_position().y1 for ax in row) + lift
    for ax, (label, x_offset) in zip(row, labels_and_offsets):
        box = ax.get_position()
        ax.text(x_offset, (row_top - box.y0) / box.height, label,
                transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
                fontweight="bold", va="bottom", ha="left")

stem = f"channel_breakdown_{PRIMARY_REFERENCE}"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

In [ ]:
# Same frame as figure 1 -- 6.27 in text width, same fonts, same panel markers -- so the
# two sit in the document at the same physical text size.
PROFILES_XLIM = (0.2, 500)
LOG_RMS = False   # the bottom panel's coherence fill: False = RMS, True = log-RMS
# One per-channel panel per reference, top to bottom. The rates panel below them takes
# its RMS fill from the first.
PROFILE_REFERENCES = ("avg", "linked_mastoids")

def plot_channel_profiles(ax, reference):
    """Every channel's coherence magnitude, coloured by where it sits on the head."""
    magnitude = np.abs(pooled_grand_average(reference)["coherence"])   # (F, n_eeg)
    info = info_for(reference)
    colours = _rgb(*np.array([channel["loc"][:3] for channel in info["chs"]]).T)
    for i in range(magnitude.shape[1]):
        # 64 traces over 2561 points each: rasterised, or the PDF balloons.
        ax.plot(frequencies[1:], magnitude[1:, i],
                color=colours[i], lw=0.4, alpha=0.6, rasterized=True)
    ax.set_ylabel("Coherence magnitude")
    ax.tick_params(axis="x", labelbottom=False)
    # Figure 2's row titles, so a reference is named the same way in both figures.
    ax.set_title(REFERENCE_TITLES[reference].format(n=magnitude.shape[1]))
    return bare(ax)


def plot_sensor_key(ax, reference):
    """The head, with each sensor in the colour its trace is drawn in."""
    info = info_for(reference)
    colours = _rgb(*np.array([channel["loc"][:3] for channel in info["chs"]]).T)
    sensor_positions, outlines = _get_pos_outlines(info, np.arange(len(info.ch_names)),
                                                   sphere=(0, 0, 0, 0.095))
    _draw_outlines(ax, outlines)
    ax.scatter(*sensor_positions.T, c=colours, s=5, lw=0)
    ax.set_aspect("equal")
    ax.set_axis_off()


fig = plt.figure(figsize=(FIGSIZE[0], FIGSIZE[1] * 1.45))
# The right-hand margin is a key column: the sensor map beside the traces, the legend
# beside the rates.
gs = gridspec.GridSpec(len(PROFILE_REFERENCES) + 1, 1, figure=fig,
                       height_ratios=[5] * len(PROFILE_REFERENCES) + [4],
                       hspace=0.45, right=0.923)

# --- the per-channel panels, one per reference ------------------------------------
# Shared y as well as x, so the references are compared at one scale.
profile_axes = []
for row, reference in enumerate(PROFILE_REFERENCES):
    shared = profile_axes[0] if profile_axes else None
    ax = fig.add_subplot(gs[row], sharex=shared, sharey=shared)
    plot_channel_profiles(ax, reference)
    profile_axes.append(ax)
profile_axes[0].set_ylim(0, None)

# One key serves every panel only if they carry the same sensors; FCz drops one.
assert all(info_for(r).ch_names == info_for(PROFILE_REFERENCES[0]).ch_names
           for r in PROFILE_REFERENCES), "the panels do not share a channel set"
plot_sensor_key(profile_axes[0].inset_axes([0.58, 0.37, 0.6, 0.6]), PROFILE_REFERENCES[0])

# --- the speech-unit rates, over the RMS coherence ---------------------------------
# The RMS trace is figure 1's, mapped onto the histograms' 0-1 axis with its peak at 1.
# LOG_RMS maps it in decades instead, with the noise level at 0: on a linear scale
# everything above ~20 Hz flattens into the floor.
ax_rate = fig.add_subplot(gs[-1], sharex=profile_axes[0])
if LOG_RMS:
    noise_level = np.median(null_median[1:])
    rms_trace = np.clip(np.log10(spectrum[1:] / noise_level)
                        / np.log10(spectrum[1:].max() / noise_level), 0, None)
else:
    rms_trace = spectrum[1:] / spectrum[1:].max()

    
# ax_rate.fill_between(frequencies[1:], 0, rms_trace, color=INK, lw=0, alpha=0.25,
#                      label="Log-RMS coherence" if LOG_RMS else "RMS\ncoherence",
#                      zorder=0, rasterized=True)
plot_unit_rates(ax_rate)
ax_rate.set_title("Rates of speech units in the stimuli")
# plot_unit_rates places its legend inside figure 1's panel; here the RMS fill
# covers that spot, so it moves out into the key column.
ax_rate.legend(loc=(0.57, 0.05), frameon=False,
               fontsize=TICK_FONTSIZE, handlelength=1.2, labelspacing=0.25)

ax_rate.set_xlim(*PROFILES_XLIM)
ax_rate.set_xticks([1, 10, 100], [1, 10, 100])
# plot_unit_rates pads its label for figure 1, and align_ylabels adds each label's own
# pad on top of the shared position, so it would sit further out than the others.
ax_rate.yaxis.labelpad = plt.rcParams["axes.labelpad"]
fig.align_ylabels(profile_axes + [ax_rate])

all_axes = profile_axes + [ax_rate]
for ax, label in zip(all_axes, "abcdefg"):
    ax.text(-0.075, 1.08, f"{label})", transform=ax.transAxes,
            fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold", va="bottom", ha="left")

stem = f"channel_profiles_{'_'.join(PROFILE_REFERENCES)}"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

## Figure 4 -- participant-level coherence, low and high frequency

Everything above pools the complex coherence over *all* trials before taking a
magnitude. This figure goes one level down: one row per participant, on the
`gammatone_null_by_subject` files of `PRIMARY_REFERENCE`. Each column is one band; the
top panel is a participant x frequency heatmap and the bottom panel the
across-participant median against the null.

Each participant is put on their own scale in three steps:

1. take the participant's duration-weighted **complex-average** coherence and reduce
   it to an **RMS coherence magnitude** over the sensors, `(F,)`;
2. subtract the null's **median RMS magnitude, by frequency** -- the median over that
   participant's 500 null draws at each bin;
3. divide by the **spread of that participant's own null draws** -- the RMS of the
   centred draws, pooled over the 500 draws and every analysed bin, one scalar.

Both the centre and the scale therefore come from the participant's null, and the unit
is one null SD: a value of 3 means three times that participant's own noise spread. The
null has to be handled per participant, since its width varies several-fold across
participants and tracks listening time; dividing by it is what puts everyone on one
axis. The participant's null draws take the identical treatment, the same by-frequency
median and the same divisor, so they sit in the same units as the real trace.

Rows are comparable in **amplitude as well as shape**: a participant's height is their
signal-to-noise ratio, so someone who tracks strongly is brighter in the heatmap rather
than merely differently shaped. Scaling by the RMS of the *real* trace instead -- the
earlier version of this cell -- forced every row to unit RMS and threw that away, and
it also made the divisor grow with the very signal it was meant to normalise: a
participant who tracks well got a larger divisor, which shrank their null draws while
their real trace was pinned to unit RMS like everyone else's. Self-consistent under the
null hypothesis, but off it the bands came out around 10% narrow.

The lower panels show the **median across participants**, with the null combined the
same way: each draw is reduced to its own across-participant median, giving 500
realisations of the median trace, and the grey band is their **full range** -- the
same envelope figure 1's panel (a) shades, so both figures are read the same way --
with the grey line their median. The band is therefore the sampling spread *of the
median trace* under the null, not the spread across participants, which is much wider.

The high band is smoothed with a 25-bin (5 Hz) moving average, the low band not at
all; 30-300 Hz is 1350 bins of the 0.2 Hz grid across two inches of page and is not
legible raw. The smoothing is applied to the **whole spectrum before the band is
sliced**, so no window is zero-padded inside the plotted range, and it is applied to
the null draws on exactly the same terms as to the real data -- the median across
participants first, the moving average second, for both. Smoothing only the red trace
would leave the grey band carrying the spread of a raw-bin statistic, roughly four
times wider than the one being drawn against it.

The heatmap rows are smoothed per participant instead, which is a display choice about
legible rows and not the tested quantity: the per-participant distribution is
right-skewed, so smoothing before the median lifts it by ~30% over the high band. The
lower panel therefore sits slightly below the median of the rows above it.

The across-participant median does not take listening-time weights; step 3 already puts
every participant on an equal footing. Each heatmap is on a symmetric-log colour scale in the
same null-SD units, linear below 1 and compressed above it, floored at -1.5 so the
handful of cells further below a participant's noise floor saturate rather than
spending colour range.

The first run for a reference reads its ~51 GB of per-subject files (~9 min); after that
the reduction is read back from `figure_data/<reference>/participant_level.npz`.

In [ ]:
# Per-participant referencing and scaling, done in one pass over the per-subject null
# files and cached per reference; the plot cell below only slices bands.
PARTICIPANT_BANDS = {"low": (0.0, 30.0), "high": (30.0, 300.0)}
MEDIAN_COLOUR = "#c1272d"

# What step (3) below divides by. Written into the cache and checked on the way back
# out: a cache built under the other basis has the right shapes and the wrong units,
# so it would be read back silently.
SCALE_BASIS = "null"


def subject_null_dir(reference):
    """That reference's per-subject pooled-null directory."""
    return resolve_reference_dirs(CONFIG, reference, ROOT)[1] / "gammatone_null_by_subject"


def pool_participants(paths, analysed):
    """Each participant's RMS trace, referenced to and scaled by their own null.

    Both the centre and the scale come from that participant's own null draws, so a
    value is read as multiples of their noise spread and rows are comparable in
    amplitude as well as shape.

    Returns the normalised real traces ``(n_subjects, F)``; the null combined the same
    way as the real data -- the across-participant median, draw by draw --
    ``(n_draws, F)``; each participant's listening time; and their scale.
    """
    subject_norm, subject_listening, subject_scale = [], [], []
    null_norm_all = None

    for index, path in enumerate(tqdm(paths, desc="subjects", unit="file")):
        with np.load(path) as saved:
            # (1) complex-average coherence -> RMS magnitude over the sensors.
            real = saved["gammatone_coherence"].squeeze(1)                  # (F, n_eeg)
            real_rms = np.sqrt((np.abs(real) ** 2).mean(-1))                # (F,)
            subject_listening.append(float(saved["total_analysed_duration"]))

            # In place: a second (D, F, n_eeg) temporary is ~0.3 GB per participant.
            power = np.abs(saved["null_gammatone_coherence"].squeeze(2))    # (D, F, n_eeg)
            power **= 2
            null_rms = np.sqrt(power.mean(-1))                              # (D, F)
        del power

        if null_norm_all is None:
            # The across-participant median cannot be streamed the way a sum can, so
            # every participant's normalised null is held at once -- ~0.4 GB.
            null_norm_all = np.empty((len(paths), *null_rms.shape), np.float32)

        # (2) remove the participant's by-frequency null median. Not `null_median`:
        # that name is figure 1's pooled null, and the cells above still use it.
        own_null_median = np.median(null_rms, axis=0)                       # (F,)
        real_centred = real_rms - own_null_median

        # (3) divide by the spread of that participant's OWN null draws -- the RMS of
        # the centred draws, pooled over draws and frequency, one scalar. Taking it
        # from the real trace instead makes the divisor grow with the signal it is
        # meant to measure: a participant who tracks well gets a larger divisor, which
        # shrinks their null while their real trace is normalised to unit RMS like
        # everyone else's. That is self-consistent under the null hypothesis, but off
        # it the nulls come out ~10% narrow and the rows lose any amplitude meaning.
        # From the null, the divisor is a property of the participant's noise alone,
        # the unit is one null SD, and a row's height is its signal-to-noise ratio.
        null_centred = null_rms - own_null_median
        scale = float(np.sqrt((null_centred[:, analysed] ** 2).mean()))
        subject_scale.append(scale)
        subject_norm.append((real_centred / scale).astype(np.float32))
        null_norm_all[index] = null_centred / scale

    null_combined = np.median(null_norm_all, axis=0).astype(np.float32)
    return (np.array(subject_norm), null_combined, np.array(subject_listening),
            np.array(subject_scale))


def participant_level(reference):
    """The cached per-participant reduction for one reference, building it on first use.

    The cache carries the files it was built from: a reference whose per-subject
    directory later gains or loses a participant would otherwise be read back with the
    old rows, with the right shapes and no error.
    """
    paths = sorted(subject_null_dir(reference).glob("*.npz"))
    names = [path.stem for path in paths]
    cache = cache_dir(reference) / "participant_level.npz"
    if cache.exists():
        with np.load(cache) as saved:
            fresh = (str(saved["scale_basis"]) == SCALE_BASIS
                     if "scale_basis" in saved else False) and (
                     [str(name) for name in saved["subjects"]] == names
                     and np.array_equal(saved["frequencies"], frequencies))
            if fresh:
                return {key: saved[key] for key in
                        ("subject_norm", "null_combined", "subject_listening", "subject_scale")}
        print(f"  {reference}: cached participants, grid or scale basis differ, re-pooling")

    analysed = frequencies > 0          # DC is dropped everywhere in this notebook
    subject_norm, null_combined, subject_listening, subject_scale = pool_participants(
        paths, analysed)
    np.savez(cache, frequencies=frequencies, subjects=names, subject_norm=subject_norm,
             null_combined=null_combined, subject_listening=subject_listening,
             subject_scale=subject_scale, scale_basis=SCALE_BASIS)
    return dict(subject_norm=subject_norm, null_combined=null_combined,
                subject_listening=subject_listening, subject_scale=subject_scale)


participants = participant_level(PRIMARY_REFERENCE)
subject_norm = participants["subject_norm"]              # (n_subjects, F)
null_combined = participants["null_combined"]            # (n_draws, F)
subject_listening = participants["subject_listening"]
subject_scale = participants["subject_scale"]

analysed = frequencies > 0
print(f"{PRIMARY_REFERENCE}: {len(subject_norm)} participants, {null_combined.shape[0]} "
      f"null draws, {subject_listening.sum() / 3600:.1f} listening hours")
print(f"per-participant scale (RMS spread of their own null draws) "
      f"{subject_scale.min():.5f}-{subject_scale.max():.5f} "
      f"({subject_scale.max() / subject_scale.min():.1f}x)")
print(f"per-participant SNR (RMS of the normalised real trace) "
      f"{np.sqrt((subject_norm[:, analysed] ** 2).mean(1)).min():.2f}-"
      f"{np.sqrt((subject_norm[:, analysed] ** 2).mean(1)).max():.2f}")
print(f"null over {int(analysed.sum())} bins -- median of the median line "
      f"{np.median(np.median(null_combined[:, analysed], axis=0)):+.4f}, "
      f"spread (SD) {null_combined[:, analysed].std():.4f}")

In [ ]:
from matplotlib.colors import SymLogNorm
from matplotlib.patches import Patch
from scipy.ndimage import uniform_filter1d

# The symlog colour scale, in standardised units. LINTHRESH is where it turns over:
# below it the cells nearest the participant's own noise floor stay linear, above it
# the long positive tail is compressed. COLOUR_FLOOR cuts the negative end off rather
# than spending colour range on it -- below the noise floor is below the noise floor,
# and the few cells under it saturate.
LINTHRESH, LINSCALE, COLOUR_FLOOR = 1.0, 0.5, -1.5
COLOURBAR_TICKS = [-10, -5, -2, -1, 0, 1, 2, 5, 10, 20]

# Smoothing, in bins of the 0.2 Hz grid. The high band puts 1350 bins across two
# inches of page, so a 5 Hz window is what makes its structure legible; the low band
# is drawn raw. Whatever is applied here is applied to the null draws too -- smoothing
# only the real trace leaves the grey band holding the spread of a raw-bin statistic
# while the red holds a smoothed one, which made the band ~4x too wide here.
SMOOTH_BINS = {"low": 1, "high": 15}


def smooth(trace, width):
    """Moving average along the last axis, taken over the *whole* spectrum.

    Run on a band slice instead and the padding falls inside the plotted range: a
    zero-padded 25-bin window halved the 30 Hz and 300 Hz ends of the high band and
    manufactured most of the rise out of its left edge. Smoothing the full spectrum
    and slicing afterwards gives those bins their real neighbours, and the only
    padded bins are DC and Nyquist, neither of which is plotted.
    """
    if width <= 1:
        return trace
    return uniform_filter1d(trace, width, axis=-1, mode="nearest")

row_order = np.argsort(subject_listening)          # least listening time at the bottom

fig, axes = plt.subplots(2, 2, figsize=(FIGSIZE[0], 5.4), sharex="col",
                         gridspec_kw=dict(height_ratios=[1, 1], hspace=0.16, wspace=0.42,
                                       left=0.175, right=0.90))

for column, (name, (low, high)) in enumerate(PARTICIPANT_BANDS.items()):
    ax_heat, ax_traces = axes[0, column], axes[1, column]
    in_band = (frequencies > low) & (frequencies <= high)
    band_frequencies = frequencies[in_band]
    xlim = (low if low > 0 else band_frequencies[0], high)

    # Everything is already referenced and scaled per participant; bands are slices.
    # Smooth first and slice second, so no window straddles a band edge.
    width = SMOOTH_BINS[name]
    real = smooth(subject_norm, width)[:, in_band][row_order]  # (n_subjects, F_band)

    # The trace is the median taken first and smoothed after, and every null draw goes
    # through that identical pair of operations -- so the grey band is the null
    # distribution of exactly the statistic drawn in red. Smoothing each participant
    # first and taking the median after is a different statistic: the per-participant
    # distribution is right-skewed, so tightening it lifts the median, by ~30% over the
    # high band here. That is the order the heatmap uses, because there the smoothing
    # is a display choice about legible rows; it is not the quantity being tested.
    median_trace = smooth(np.median(subject_norm, axis=0), width)[in_band]
    null_draws = smooth(null_combined, width)[:, in_band]      # (n_draws, F_band)
    # The full range of the draws, not a central percentile: the same envelope
    # figure 1's panel (a) shades, so the two figures are read the same way, and a
    # trace clear of it is clear of every draw rather than of 99% of them. The cost
    # is a rougher boundary -- min and max are the extreme order statistics of 500
    # and jitter ~2x more per bin than the 1st/99th percentile they replace.
    null_low, null_high = null_draws.min(0), null_draws.max(0)

    # Symmetric-log colour: the values are signed -- negative is below that
    # participant's own noise floor -- and the positive tail runs to ~23 null SDs
    # while the bulk sits near zero. A linear scale spends its range on the tail; a plain log
    # cannot take the negatives. LINTHRESH keeps the noise-floor band linear (a log
    # scale has no room near zero) and compresses everything beyond it, so the full
    # range can be shown without clipping to percentiles.
    mesh = ax_heat.pcolormesh(
        band_frequencies, np.arange(len(real)), real, shading="auto", cmap="viridis",
        rasterized=True,
        norm=SymLogNorm(linthresh=LINTHRESH, linscale=LINSCALE,
                        vmin=COLOUR_FLOOR, vmax=real.max()),
    )
    ax_heat.set_xlim(*xlim)
    ax_heat.set_yticks([])
    ax_heat.set_title(f"{low:g}-{high:g} Hz")

    # An inset colourbar, not fig.colorbar: the latter narrows ax_heat to make room, and
    # ax_traces below -- a separate gridspec cell -- would no longer line up with it,
    # since sharex syncs the data range and not the width of the box. No extend either:
    # its end blocks leave a seam in the PDF. Cells outside vmin/vmax simply saturate.
    colourbar = fig.colorbar(mesh, cax=ax_heat.inset_axes([1.03, 0.0, 0.04, 1.0]))
    colourbar.ax.tick_params(labelsize=TICK_FONTSIZE)
    colourbar.outline.set_visible(False)
    # Explicit ticks: the symlog locator labels every decade of the log region and
    # nothing in the linear one, which leaves the band the data actually sits in blank.
    ticks = [t for t in COLOURBAR_TICKS if COLOUR_FLOOR <= t <= real.max()]
    colourbar.set_ticks(ticks, labels=[f"{t:g}" for t in ticks])

    # The null goes over the trace, not under it: where the red disappears into the
    # grey it is inside the range of the null draws, which is the thing worth seeing.
    ax_traces.plot(band_frequencies, median_trace,
                   color=MEDIAN_COLOUR, lw=1.0, zorder=1)
    ax_traces.fill_between(band_frequencies, null_low, null_high,
                           color=NULL_FILL, alpha=0.75, lw=0, zorder=2)
    ax_traces.plot(band_frequencies, np.median(null_draws, axis=0),
                   color=MUTED, lw=0.8, zorder=3)
    ax_traces.set_xlim(*xlim)
    ax_traces.set_xlabel("Frequency (Hz)")
    bare(ax_traces)

axes[0, 0].set_ylabel("Participant\n(ordered by listening time)")
# Both axes are now literally in null SDs, so the label says which.
axes[1, 0].set_ylabel("Standardised RMS coherence", labelpad=5)
fig.align_ylabels(axes[:, 0])

# Into the low panel's empty upper-right corner -- the high panel's is under the peak.
axes[1, 0].legend(
    handles=[
        Line2D([0], [0], color=MEDIAN_COLOUR, lw=1.0),
        Line2D([0], [0], color=MUTED, lw=0.8),
        Patch(facecolor=NULL_FILL, alpha=0.75),
    ],
    labels=["Median across participants", "Null median",
            f"Null range ({len(null_draws)} draws)"],
    loc=(0.03, 0.7), fontsize=TICK_FONTSIZE, frameon=False,
)

fig.suptitle(f"Participant-level EEG-envelope coherence, "
             f"{REFERENCE_LABELS[PRIMARY_REFERENCE].lower()}", fontsize=DEFAULT_FONTSIZE)

for ax, label in zip(axes.ravel(), "abcd"):
    ax.text(-0.17, 1.03, f"{label})", transform=ax.transAxes,
            fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold", va="bottom", ha="left")

stem = f"participant_level_coherence_{PRIMARY_REFERENCE}"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

## Figure 5 -- coherence averaged across subjects

Everything above pools on the **complex** coherence and takes the magnitude once, at
the end. That lets a component present in every subject at an inconsistent phase
cancel, and it is why the null there sits so low: the surrogate coherences cancel too.

This section takes the other order of operations. Each subject's own duration-weighted
complex coherence is formed first -- by `get_null_gammatone_coherence.py`'s per-subject
grouping, not here -- and its magnitude is taken per subject. The subjects are then combined by a
duration-weighted **mean of those magnitudes**, `mean(|C_s|)`. `SUBJECT_COMBINATION`
in the cell below switches that one axis to an RMS, `sqrt(mean(|C_s|**2))` -- the rule
the notebook uses everywhere else -- and caches the two separately so they can be
compared without re-reading 51 GB each time.

Only the subject axis is at stake. The reduction over frequency bins within a band,
and the one over the 64 channels that makes the trace, stay RMS as they are throughout
this notebook, so the y-axis is figure 1's and means the same thing.

Nothing is standardised per subject first -- that is figure 4's construction, and a
different question. Here each subject enters on their own raw scale, weighted by
listening time, so a subject with strong coherence contributes more than a weak one.

Two consequences worth keeping in view:

- RMS is never smaller than the plain mean of the same magnitudes (Jensen), so the
  RMS setting lifts both the trace and its null, and more so the more subjects differ
  from each other. Measured here it is a small, near-common shift: the peak goes 0.064
  to 0.072 and the null median 0.0176 to 0.0200, while the per-band elevations move by
  under three points (161% against 158% in the lowest band). The choice moves the axis,
  not the finding.
- Nothing cancels in either direction. The estimate answers *how strong is coherence
  within a subject*, not *how consistent is it across subjects*. Its null is put
  through the identical RMS combination, so the comparison stays like-for-like -- but
  that null is very tight, because the band average runs over hundreds of bins and all
  78 subjects, so a few per cent of elevation is already many standard deviations. The
  estimator is sensitive rather than specific, and the per-band sensor counts printed
  below are the check on that rather than something to assert.

Panel (b) puts the two orders of operations against each other. The grand average of
figure 1 takes the magnitude last, so anything carried at an inconsistent phase across
subjects cancels before it is measured; panel (a)'s estimator takes each subject's
magnitude first, so nothing cancels. Their ratio,
`|grand average| / mean_s |C_s|`, is what that cancellation costs: 1 where every
subject carries the same phase and chance where none of them do. Drawn bin by bin,
the trace reaches 0.72 at 0.6 Hz and again at 8.0 Hz, dips to 0.23 at 3.6 Hz, holds
0.58 at 44 Hz, and is within half again of chance above about 220 Hz.

Chance is not 0 -- the line is labelled *Noise level* in the panel, as figure 1 labels
its own. Averaging 81 subjects' phases leaves a residue of order
`1 / sqrt(81)`, so the level is measured rather than assumed: the same ratio, formed
draw by draw from the two nulls the other panels already carry, which comes out flat
at 0.124 across the whole spectrum -- 0.123 to 0.124 in every band from 0.2 Hz to
500 Hz. That is a shade above `1 / sqrt(78.1)` = 0.113, the duration-weighted
effective N, because the numerator's RMS over channels sees the full Rayleigh spread
of a complex average while the denominator has already averaged that spread away over
the subjects. A simulation of 81 complex-Gaussian subjects on the real listening
times, put through both panels' reductions, gives 0.126 against the measured 0.124, so
nothing in that level is signal. Only the median of the 500 draws is drawn by default: their
spread is wide (0.046 to 0.309 over bins and draws), so shading it says more about how
noisy a single surrogate ratio is than about the trace. `CONSISTENCY_NULL_RANGE`
shades it anyway, as panel (a) shades its own null.

Four options govern the panel. `CONSISTENCY_REDUCTION` sets how the trace collapses
the sensors: RMS is the notebook's reduction everywhere else and is exactly the
statistic the chance line is formed from -- panel (b) is then figure 1's panel (a)
divided by this figure's -- so the two are like for like; `"mean"`, the plain mean of
the per-sensor ratios, runs about five points lower and is not the null's statistic.
`CONSISTENCY_CHANNEL_TRACES` draws each sensor's own ratio faintly behind the trace,
coloured by position on the head exactly as the channel-profiles figure colours its
channels and keyed by the same head, so a colour means the same thing in both.
`CONSISTENCY_SMOOTHING` is 0, so every bin is drawn as it is; it takes a window
proportional to frequency rather than fixed in bins, and is applied to the null draws
as well as to the trace.

The peak frequencies are fitted from *this* spectrum, with the same quartic in
log-log space figure 1 uses, so the stems mark where this estimator actually peaks
rather than inheriting figure 1's.

The topographies come out flatter than figure 1's, and that is the estimator rather
than the drawing: the null level here is 0.0176 against a typical band magnitude of
0.0229, so roughly three quarters of every sensor's value is the same noise floor,
which compresses the contrast between sensors. Across the scalp the coefficient of
variation runs 4-19% by band where figure 1's runs 27-42%. `SUBTRACT_NOISE_FLOOR` in
the figure cell takes each sensor's own median noise level off first, which is what
the maps are drawn with by default; it lifts the spread to 26-46% and no sensor goes
negative in any band. `SUBJECT_HIDDEN_BANDS` drops 20-26 Hz from the panel, where it
is squeezed between its neighbours -- the cell below still pools and reports it.

Nothing is masked for significance, for the reason the counts give: every sensor
clears both thresholds in every band, so dots would cover each map without
distinguishing anything.

In [ ]:
# One pass over the per-subject files, cached per reference beside the other caches.
# Reads the same ~51 GB figure 4 does, and takes about the same time.

# How the subjects are combined, once each has its own magnitude. "mean" averages the
# magnitudes, `mean(|C_s|)`; "rms" averages their squares and roots the result,
# `sqrt(mean(|C_s|**2))`. RMS is never the smaller of the two (Jensen), and the gap
# between them widens the more subjects differ from each other, so the pair is worth
# flipping between. Only the subject axis is at stake: the reduction over frequency
# bins within a band, and the one over the 64 channels, stay RMS as everywhere else in
# this notebook -- figure 1 collapses its channels the same way.
SUBJECT_COMBINATION = "mean"      # "mean" | "rms"


def pool_subject_average(paths, bands, combination=None):
    """Combine the subjects' own coherence magnitudes, and their nulls with them.

    The null goes through the identical order, so the two stay comparable. Returns the
    per-sensor magnitude ``(F, n_eeg)``; the null's RMS-over-sensors spectra
    ``(D, F)``; the per-band null kept per channel ``(D, n_bands, n_eeg)``; and the
    listening time behind them.
    """
    square = (SUBJECT_COMBINATION if combination is None else combination) == "rms"
    real_power = null_power = scratch = None
    weight_sum = 0.0

    for path in tqdm(paths, desc="subjects", unit="file"):
        with np.load(path) as saved:
            weight = np.float32(saved["total_analysed_duration"])
            # The magnitude is taken per subject either way -- that is what makes this
            # the other order of operations. Squaring first is what separates the two
            # combinations, so it is the only thing the option changes.
            subject_power = np.abs(saved["gammatone_coherence"].squeeze(1))
            if square:
                subject_power **= 2
            raw_null = saved["null_gammatone_coherence"].squeeze(2)   # (D, F, n_eeg)

            if real_power is None:
                axis = saved["frequencies"]
                real_power = np.zeros(subject_power.shape, np.float64)
                null_power = np.zeros(raw_null.shape, np.float32)
                scratch = np.empty(raw_null.shape, np.float32)

            # In place: |null| ** 2 on a (500, F, n_eeg) complex64 block allocates
            # 328 MB per temporary, and there would be two of them per subject.
            np.abs(raw_null, out=scratch)
            if square:
                scratch **= 2
            scratch *= weight
            null_power += scratch

            real_power += subject_power * float(weight)
            weight_sum += float(weight)

    real_power /= weight_sum
    null_power /= np.float32(weight_sum)
    if square:
        np.sqrt(real_power, out=real_power)
        np.sqrt(null_power, out=null_power)
    real_rms = real_power.astype(np.float32)      # (F, n_eeg) magnitude, either way
    # From here the null is a magnitude too, so the band and channel reductions below
    # square it themselves rather than relying on how it was accumulated.
    null_power **= 2

    masks, _ = band_bin_masks(axis, bands)
    # Kept per channel rather than collapsed to the max: the per-sensor null is what
    # lets "every channel is significant" be checked instead of asserted.
    band_null = np.stack([null_power[:, mask, :].mean(1) for mask in masks],
                         axis=1).astype(np.float32)               # (D, n_bands, n_eeg)
    null_spectra = np.sqrt(null_power.mean(-1)).astype(np.float32)  # (D, F)
    return axis, real_rms, null_spectra, band_null, weight_sum


def subject_magnitude_average(reference):
    """The cached across-subjects reduction for one reference, under the option above.

    Carries both the subjects it was built from and the bands it was reduced over:
    ``band_null`` is one slab per band, so a cache written for other bands would hand
    back thresholds for the wrong frequency ranges, with the right shape and no error.
    """
    bands = bands_for(reference)
    paths = sorted(subject_null_dir(reference).glob("*.npz"))
    names = [path.stem for path in paths]
    # The combination is in the filename, so the two live side by side and flipping
    # the option back does not re-read 51 GB for something already on disk.
    cache = cache_dir(reference) / f"subject_{SUBJECT_COMBINATION}_coherence.npz"

    if cache.exists():
        with np.load(cache) as saved:
            if ([str(name) for name in saved["subjects"]] == names
                    and np.array_equal(saved["bands"], bands)):
                return {key: saved[key] for key in
                        ("frequencies", "coherence", "null_spectra", "band_null")} | dict(
                    listening_seconds=float(saved["listening_seconds"]), subjects=names)
        print(f"  {reference}: cached subjects or bands differ, re-pooling")

    axis, coherence_, null_spectra_, band_null_, seconds = pool_subject_average(paths, bands)
    np.savez(cache, frequencies=axis, coherence=coherence_, null_spectra=null_spectra_,
             band_null=band_null_, listening_seconds=seconds, bands=bands, subjects=names,
             combination=SUBJECT_COMBINATION)
    return dict(frequencies=axis, coherence=coherence_, null_spectra=null_spectra_,
                band_null=band_null_, listening_seconds=seconds, subjects=names)


subject_average = subject_magnitude_average(PRIMARY_REFERENCE)
subject_coherence = subject_average["coherence"]                  # (F, n_eeg)
subject_null_spectra = subject_average["null_spectra"]            # (D, F)
subject_band_null = subject_average["band_null"]                  # (D, n_bands, n_eeg)

subject_spectrum = np.sqrt((subject_coherence ** 2).mean(-1))     # (F,)
subject_null_range = (subject_null_spectra.min(0), subject_null_spectra.max(0))
subject_null_median = np.median(subject_null_spectra, axis=0)

# Peaks fitted from this spectrum, not inherited from figure 1's.
SUBJECT_BAND_CENTRES, subject_peak_is_interior = [], []
for index, bins in enumerate(band_masks_by_frequency):
    peak, _, interior = band_peak(subject_spectrum, bins)
    SUBJECT_BAND_CENTRES.append(snap(peak, index))
    subject_peak_is_interior.append(interior)
SUBJECT_BAND_CENTRES = np.array(SUBJECT_BAND_CENTRES)

# The same two tests figure 1's topographies use, on this estimator's own null. Both
# sides are in power, so the ratio reported as an elevation is square-rooted back into
# the magnitude units the y-axes carry.
per_channel_quantile = 100 * (1 - ALPHA / (len(BANDS) * subject_coherence.shape[1]))
subject_power_ = subject_coherence ** 2
subject_topographies, subject_masks, subject_rows = [], [], []
for index, bins in enumerate(band_masks_by_frequency):
    observed = subject_power_[bins].mean(0)                       # (n_eeg,)
    band_null_here = subject_band_null[:, index, :]               # (D, n_eeg)
    max_threshold = np.percentile(band_null_here.max(1), quantile)
    channel_threshold = np.percentile(band_null_here, per_channel_quantile, axis=0)
    subject_topographies.append(np.sqrt(observed))
    subject_masks.append(observed > max_threshold)
    subject_rows.append((
        int((observed > max_threshold).sum()),
        int((observed > channel_threshold).sum()),
        float(np.median(np.sqrt(observed / np.median(band_null_here, 0))) - 1),
        float(np.median((observed - band_null_here.mean(0)) / band_null_here.std(0))),
    ))

print(f"{len(subject_average['subjects'])} subjects combined by "
      f"{SUBJECT_COMBINATION}, "
      f"{subject_average['listening_seconds'] / 3600:.1f} listening hours")
print(f"peak {subject_spectrum[1:].max():.3f} at "
      f"{frequencies[1:][np.argmax(subject_spectrum[1:])]:.1f} Hz; null median "
      f"{np.median(subject_null_median):.4f} "
      f"(complex-average null median {np.median(null_median):.4f})")
print(f"\n{'band':>14}  {'peak':>7}  {'max-stat':>8} {'per-chan':>8}  "
      f"{'elev(mag)':>9} {'median z':>9}")
for (low, high), centre, interior, row in zip(
        BANDS, SUBJECT_BAND_CENTRES, subject_peak_is_interior, subject_rows):
    n_max, n_channel, elevation, z = row
    edge = "" if interior else "  (edge)"
    print(f"{low:6.1f}-{high:5.1f} Hz  {centre:6g}{edge}  {n_max:5d}/{len(INFO.ch_names)} "
          f"{n_channel:5d}/{len(INFO.ch_names)}  {elevation * 100:8.1f}% {z:9.1f}")

In [ ]:
from matplotlib.ticker import NullFormatter
# The position-to-colour map and the head outline panel (b)'s key is drawn
# with. Private MNE helpers, as in the channel-profiles figure -- there is no
# public route to either.
from mne.viz.evoked import _rgb
from mne.viz.topomap import _draw_outlines, _get_pos_outlines

# Figure 1's panel (a), on the other order of operations. The window has to be taken
# from the data: this estimator and its null both sit well above the complex-average
# ones -- the null median is ~10x higher -- so figure 1's YLIM would not hold it, and
# the trace spans barely a decade where figure 1's spans nearly three.
#
# The *proportions* are still figure 1's, though. The headroom above the peak is
# measured off that panel as a fraction of its height and reproduced here, so the
# trace tops out at the same place under the topographies and the two panels are read
# at one glance. Fixing a multiplier by hand instead would drift the moment either
# figure's window moved.
SUBJECT_HEADROOM = np.log10(YLIM[1] / spectrum[1:].max()) / np.log10(YLIM[1] / YLIM[0])

_floor = np.log10(subject_null_range[0][1:].min() * 0.75)
_peak = np.log10(subject_spectrum[1:].max())
SUBJECT_YLIM = (10 ** _floor,
                10 ** ((_peak - SUBJECT_HEADROOM * _floor) / (1 - SUBJECT_HEADROOM)))
SUBJECT_INSET_Y = INSET_Y     # the same row as figure 1, which is what the above buys
SUBJECT_STEM_LABEL_Y = SUBJECT_YLIM[0] * 1.1

# --- two options for the topographies ----------------------------------------
# What each map shows. False is the band's RMS coherence magnitude as it stands, which
# is three quarters noise floor and comes out nearly uniform; True takes each sensor's
# own median noise level off first, leaving what the band carries above it. The maps
# have no colourbar, so this is a change of contrast rather than of colour scale, but
# it is a different quantity and the saved file is named accordingly.
#
# The subtraction is in magnitude, as both terms are drawn: sqrt of the band-mean power
# minus sqrt of its null median. Doing it in power instead -- sqrt(observed - floor),
# which is the additive one, noise power adding to signal power -- is the better
# estimate of what the band carries; it is one line below if that is wanted. The two
# rank sensors differently, because the floor is not identical across the scalp.
SUBTRACT_NOISE_FLOOR = True

# Bands left out of this panel only. 20-26 Hz is narrow and sits between the 4-15 and
# 30-60 insets, where its stem and label are hard to read. Dropping it here changes
# nothing upstream: the cell above still pools and reports every band.
SUBJECT_HIDDEN_BANDS = {(20.0, 26.0)}

shown_bands = [i for i, band in enumerate(BANDS)
               if tuple(band) not in SUBJECT_HIDDEN_BANDS]
# Laid out for the bands actually drawn, not for all of them -- otherwise dropping one
# leaves its slot empty and the rest crowded to one side.
SUBJECT_INSET_X, SUBJECT_INSET_SIZE = inset_layout(len(shown_bands))


def subject_topography(index):
    """The band's per-sensor value, as the option above asks for it."""
    magnitude = subject_topographies[index]
    if not SUBTRACT_NOISE_FLOOR:
        return magnitude
    # sqrt of the median power is exactly the median of the magnitudes -- sqrt is
    # monotone -- so this really is the sensor's median noise level.
    return magnitude - np.sqrt(np.median(subject_band_null[:, index, :], axis=0))


def log_ticks(low, high, ladder=(1.0, 2.0, 5.0)):
    """A 1-2-5 tick ladder inside a log window, as values and their labels.

    Whole decades alone are not enough here: the window is fitted to the data and
    turns out to span about one decade, which leaves a decade locator labelling a
    single tick. The ladder is only consulted where it is needed -- with three or
    more decades inside the window it falls back to decades, as figure 1 has.
    """
    decades = np.arange(np.floor(np.log10(low)), np.ceil(np.log10(high)) + 1)
    subdivisions = ladder if np.log10(high / low) < 3 else (1.0,)
    values = np.array([base * 10.0 ** decade
                       for decade in decades for base in subdivisions])
    values = values[(values >= low) & (values <= high)]
    return values, [f"{value:g}" for value in values]


# How the estimator is named, in the panel and in the title, from the one option that
# decides it -- so the words cannot say RMS while the array holds a mean.
COMBINATION_WORD = {"mean": "Mean", "rms": "RMS"}[SUBJECT_COMBINATION]


def plot_subject_spectrum(ax, null_median_line=True):
    """Subjects' own coherence magnitudes combined, over that estimator's own null.

    `null_median_line=False` leaves out the null's median line and keeps only its range, as
    figure 1's panel draws its own null.
    """
    # The full extent of the null, as one region -- figure 1's panel (a) shades the
    # same thing the same way, so the two panels are read alike.
    ax.fill_between(frequencies[1:], *[bound[1:] for bound in subject_null_range],
                    color=NULL_FILL, alpha=0.6, lw=0, zorder=1)
    if null_median_line:
        ax.plot(frequencies[1:], subject_null_median[1:], color=MUTED, lw=0.8, zorder=3)
    ax.plot(frequencies[1:], subject_spectrum[1:], color=INK, lw=0.8, zorder=0)

    # Lettered in place rather than in a legend, as in figure 1.
    ax.text(0.10, subject_spectrum[1] * 0.9, "Mean absolute\ncoherence",
            color=INK,
            fontsize=TICK_FONTSIZE, va="top", linespacing=0.9)
    ax.text(0.10, SUBJECT_YLIM[0] * 1.55, "Noise\nlevel", color=MUTED,
            fontsize=TICK_FONTSIZE, va="bottom", linespacing=0.9)

    # Still RMS, and still figure 1's label: that reduction is over the 64 channels,
    # which this section does not change. The subject combination is named in the
    # panel and in the title instead.
    ax.set_ylabel("RMS coherence magnitude")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(*XLIM)
    ax.set_ylim(*SUBJECT_YLIM)
    ax.set_xticks([0.1, 1, 10, 100], [0.1, 1, 10, 100], fontsize=TICK_FONTSIZE)
    # Derived from the limits rather than written down: the window is fitted to the
    # data, so which ticks fall inside it is not known when this is written.
    ax.set_yticks(*log_ticks(*SUBJECT_YLIM), fontsize=TICK_FONTSIZE)
    ax.yaxis.set_minor_formatter(NullFormatter())
    bare(ax)

    # Labels are laid out over the bands that are drawn: a hidden band must not
    # reserve space, or push its neighbour up a line to clear something not there.
    stem_label_text, stem_label_lifted = stem_labels(ax, SUBJECT_BAND_CENTRES[shown_bands])
    for slot, i in enumerate(shown_bands):
        band, centre = BANDS[i], SUBJECT_BAND_CENTRES[i]
        axins = ax.inset_axes(
            [SUBJECT_INSET_X[slot], SUBJECT_INSET_Y, SUBJECT_INSET_SIZE,
             SUBJECT_INSET_SIZE], transform=ax.transAxes)
        # No significance markers. Every sensor clears both thresholds in every band
        # (the counts printed above), so dots would cover each map without
        # distinguishing anything -- this estimator is sensitive, not specific.
        plot_topomap(subject_topography(i), INFO, cmap="Reds", contours=0,
                     sensors=False, show=False, axes=axins)
        low, high = band
        low_label = f"{low:.1f}" if low < 1 else f"{int(low)}"
        axins.set_title(f"{low_label}-{int(high)} Hz", fontsize=TICK_FONTSIZE)

        elbow = float(subject_spectrum[np.argmin(np.abs(frequencies - centre))])
        ax.vlines(centre, SUBJECT_YLIM[0], elbow, color=INK, lw=0.8,
                  linestyles="dotted", zorder=5)
        ax.annotate(stem_label_text[slot], xy=(centre * 1.04, SUBJECT_STEM_LABEL_Y),
                    xycoords="data", textcoords="offset points",
                    xytext=(0, STEM_LINE_POINTS if stem_label_lifted[slot] else 0),
                    color=INK, fontsize=SMALL_FONTSIZE, ha="left", va="baseline")

        anchor = ax.transData.inverted().transform(ax.transAxes.transform(
            (SUBJECT_INSET_X[slot] + SUBJECT_INSET_SIZE / 2, SUBJECT_INSET_Y)))
        ax.plot([centre, anchor[0]], [elbow, anchor[1]],
                color=INK, lw=0.8, ls="dotted", zorder=5)

    return ax


# --- the middle panel: what survives averaging across subjects ---------------------
# Panel (a)'s estimator takes each subject's magnitude first, so nothing cancels. The
# grand average of figure 1 takes the magnitude last, so a component carried at an
# inconsistent phase across subjects cancels before it is measured. Their ratio is
# what that cancellation costs, bin by bin and sensor by sensor: 1 where every subject
# carries the same phase, and chance where none of them do.
consistency = np.abs(coherence) / subject_coherence          # (F, n_eeg)

# The chance level, from the two nulls the panels above already carry. Both are RMS
# over the sensors, and the two sets of draws are independent -- one pools surrogates
# over stimuli, the other over subjects -- so pairing them by index is arbitrary, but
# each pair is still one draw of the ratio under the null, which is all it is used as.
consistency_null = null_spectra / subject_null_spectra       # (D, F)

# Which reduction over the 64 channels the drawn trace uses. "rms" is the notebook's
# reduction everywhere else and is exactly the statistic the chance line is formed from
# -- it is figure 1's panel (a) divided by this figure's -- so the trace and its chance
# level are like for like. "mean" is the plain mean of the per-sensor ratios, which
# runs some five points lower (0.64 against 0.70 in the lowest band, by Jensen) and is
# not the null's statistic.
CONSISTENCY_REDUCTION = "rms"        # "rms" | "mean"

# Every sensor's own ratio behind the trace, in the colours the channel-profiles
# figure draws its channels in -- position on the head mapped to colour -- with that
# figure's key beside them, so a colour means the same thing in both.
CONSISTENCY_CHANNEL_TRACES = True
CONSISTENCY_KEY_BOX = [0.65, 0.5, 0.5, 0.5]   # the head, in axis coordinates

# The full extent of the 500 null ratios behind their median, shaded as panel (a)
# shades its own null. Off by default: a single surrogate ratio is a noisy thing --
# the draws run 0.046 to 0.309 where their median sits at 0.124 -- so the band is
# several times wider than any uncertainty on the level the panel is read against,
# and it is that level, not the width, that the trace is compared with.
CONSISTENCY_NULL_RANGE = True

# Smoothing halfwidth, as a fraction of the centre frequency, or 0 for none -- which
# is what this panel is drawn with: the bin-to-bin scatter is part of what the sensor
# traces show. Left in place because one window in bins cannot serve this axis if it
# is ever wanted: the grid is linear at 0.2 Hz and the axis is logarithmic, so 1 Hz is
# five bins of the lowest band and a third of a bin's worth of the drawing at 300 Hz.
# Proportional to frequency, the window spans a single bin below ~1.7 Hz and hundreds
# of bins a decade up, where that many fall inside one point of the page anyway.
CONSISTENCY_SMOOTHING = 0.0


def log_smooth(trace, halfwidth=CONSISTENCY_SMOOTHING, axis_hz=None):
    """Moving average over a window that is a fixed fraction of the centre frequency.

    Taken over the whole spectrum, as the participant panels' `smooth` is, so the
    plotted bins average their real neighbours rather than the padding.
    """
    if halfwidth <= 0:
        return trace
    axis_hz = frequencies if axis_hz is None else axis_hz
    low = np.searchsorted(axis_hz, axis_hz * (1 - halfwidth), side="left")
    high = np.searchsorted(axis_hz, axis_hz * (1 + halfwidth), side="right")
    # In float64: the running sum reaches a few hundred by the top of the axis, and
    # differencing two of those in the float32 the coherences are stored at loses the
    # last digits of a window's mean.
    cumulative = np.concatenate(
        [np.zeros(trace.shape[:-1] + (1,)), np.cumsum(trace, axis=-1, dtype=float)],
        axis=-1)
    return (cumulative[..., high] - cumulative[..., low]) / (high - low)


# Put through the window once here rather than at the point of drawing, and the null
# through the identical one: smoothing the trace alone would leave its chance level
# holding the scatter of a raw-bin statistic. At the default halfwidth of 0 all three
# pass through untouched, and the arrays keep their names.
consistency_smoothed = log_smooth(consistency.T).T                    # (F, n_eeg)
consistency_null_smoothed = log_smooth(consistency_null)              # (D, F)
consistency_trace = (np.sqrt((consistency_smoothed ** 2).mean(-1))
                     if CONSISTENCY_REDUCTION == "rms" else consistency_smoothed.mean(-1))
consistency_chance = np.median(consistency_null_smoothed, axis=0)

print(f"phase consistency ({CONSISTENCY_REDUCTION} over channels): "
      f"{consistency_trace[1:].max():.2f} at "
      f"{frequencies[1:][np.argmax(consistency_trace[1:])]:.1f} Hz, "
      f"chance {np.median(consistency_chance):.3f}")


def plot_phase_consistency(ax):
    """How much of a subject's coherence survives the average across subjects."""
    if CONSISTENCY_CHANNEL_TRACES:
        colours = _rgb(*np.array([channel["loc"][:3] for channel in INFO["chs"]]).T)
        for i in range(consistency_smoothed.shape[1]):
            # 64 traces over 2560 bins each: rasterised, as the channel-profiles
            # figure rasterises the same traces, or the PDF carries 160k vector
            # points for this panel alone.
            ax.plot(frequencies[1:], consistency_smoothed[1:, i], color=colours[i],
                    lw=0.4, alpha=0.2, rasterized=True, zorder=0)
        # The same key, from the same function, so nothing can drift between them.
        plot_sensor_key(ax.inset_axes(CONSISTENCY_KEY_BOX), PRIMARY_REFERENCE)

    if CONSISTENCY_NULL_RANGE:
        # Between the two: over every real trace, the black one included, since
        # underneath them the dense high-frequency traces show through and it stops
        # reading as one region -- but under the median line it is the extent of, so
        # that line stays the thing the panel is read against.
        ax.fill_between(frequencies[1:], consistency_null_smoothed.min(0)[1:],
                        consistency_null_smoothed.max(0)[1:], color=NULL_FILL,
                        alpha=0.6, lw=0, zorder=4.5)

    ax.plot(frequencies[1:], consistency_trace[1:], color=INK, lw=0.8, zorder=4)
    # Over everything, the black trace included: it is the level the whole panel is
    # read against, and the sensor traces cross it often enough at the top of the
    # axis to swallow it underneath them.
    ax.plot(frequencies[1:], consistency_chance[1:], color=MUTED, lw=0.8, zorder=5)

    # Figure 1's wording for the same thing, so the two panels are read alike.
    ax.text(0.1, 0.0, "Noise\nlevel", color=MUTED, fontsize=TICK_FONTSIZE, va="bottom")
    ax.text(0.1, 0.40, "Phase\nconsistency", color=INK, fontsize=TICK_FONTSIZE, va="bottom")

    # Short, and on the same line as the other two panels' labels (aligned below):
    # the panel is a quarter of (a)'s height and a longer label would run into (c)'s.
    ax.set_ylabel("Consistency")
    ax.set_xscale("log")
    ax.set_xlim(*XLIM)
    ax.set_ylim(0, 1.0)
    ax.set_xticks([0.1, 1, 10, 100], [0.1, 1, 10, 100], fontsize=TICK_FONTSIZE)
    ax.set_yticks([0, 0.5, 1.0], ["0", "0.5", "1"], fontsize=TICK_FONTSIZE)
    return bare(ax)


# Three rows now, and the figure grows by exactly the new row's share of them: at the
# same height per unit of `height_ratios` panel (a) keeps the height it has in figure
# 1, rather than being squeezed to make room for the one below it.
CONSISTENCY_ROWS = [10, 5, 4]

fig = plt.figure(figsize=(FIGSIZE[0], FIGSIZE[1] * sum(CONSISTENCY_ROWS) / 14))
gs = gridspec.GridSpec(3, 1, figure=fig, height_ratios=CONSISTENCY_ROWS, hspace=0.42)

ax_subjects = fig.add_subplot(gs[0])
ax_consistency = fig.add_subplot(gs[1], sharex=ax_subjects)
ax_rates = fig.add_subplot(gs[2], sharex=ax_subjects)
for ax in (ax_subjects, ax_consistency):
    ax.tick_params(axis="x", labelbottom=False)
ax_rates.tick_params(axis="x", labelbottom=True)

plot_subject_spectrum(ax_subjects)
plot_phase_consistency(ax_consistency)
plot_unit_rates(ax_rates)      # the same rates panel as figure 1 -- the same stimuli

ax_subjects.set_title(f"Subject coherence magnitudes, combined by "
                      f"{COMBINATION_WORD.lower()}", pad=16)
ax_consistency.set_title("Consistency of coherence phase across subjects")
ax_rates.set_title("Rates of speech units in the stimuli")

for ax, label, y in [(ax_subjects, "a)", 1.06), (ax_consistency, "b)", 1.12),
                     (ax_rates, "c)", 1.12)]:
    ax.text(-0.075, y, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")
# The three labels are different lengths, so without this they sit at three different
# depths and the panel letters above them no longer line up either.
fig.align_ylabels([ax_subjects, ax_consistency, ax_rates])

# The options are in the name, so a run under one cannot overwrite the other's output.
stem = (f"subject_{SUBJECT_COMBINATION}_coherence_{PRIMARY_REFERENCE}"
        f"{'_excess' if SUBTRACT_NOISE_FLOOR else ''}"
        f"{'' if len(shown_bands) == len(BANDS) else f'_{len(shown_bands)}bands'}"
        f"{'' if CONSISTENCY_REDUCTION == 'rms' else '_meanconsistency'}")
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

### Figure 5, stacked version -- the two orders of operations, and nothing else

The two estimators figure 5 compares, each in a panel of its own over its own null, on a shared
frequency axis. Each panel keeps its band topographies, but there is no consistency ratio and no
rates panel.

- **Top: ⟨|C_xy|⟩.** Each subject's magnitude is taken first, then combined over subjects by
  `SUBJECT_COMBINATION`. This is figure 5's panel (a), so nothing cancels.
- **Bottom: |⟨C_xy⟩|.** The complex coherence is averaged over all trials first and the magnitude
  is taken last. This is figure 1's spectrum, so anything carried at an inconsistent phase across
  subjects cancels.

Both are RMS over the 64 channels, as everywhere else in this notebook. Each panel is drawn by the
function its own figure uses, so the topographies are figure 5's (noise floor subtracted when
`SUBTRACT_NOISE_FLOOR` is set) and figure 1's (drawn here without its significance dots, as figure 5's have none). Each panel also keeps
its own y window, because the stems and inset leaders are laid out against it. Neither panel draws
the null median, only the shaded range of the null.

In [ ]:
# ===================== stacked-figure options ================================
# Each panel keeps the height figure 1's panel (a) has in figure 1 -- 10 of its 14 units --
# so the topographies come out the size they are there.
STACK_FIGSIZE = (FIGSIZE[0], 2 * FIGSIZE[1] * 10 / 14)
STACK_HSPACE = 0.30            # room for (b)'s title under (a)'s axis
# =============================================================================

# The two panels are drawn by the functions figures 5 and 1 draw them with, topographies,
# stems and significance markers included, so nothing here can drift from those figures.
# Each keeps its own y window: the stems, labels and inset leaders are laid out against it.
stack_fig, (ax_magnitudes, ax_complex) = plt.subplots(
    2, 1, figsize=STACK_FIGSIZE, sharex=True, gridspec_kw={"hspace": STACK_HSPACE})

# Ranges only, no null median, in both panels: figure 1's panel already draws its null that way.
plot_subject_spectrum(ax_magnitudes, null_median_line=False)   # <|C_xy|>: per subject, then combined
# No significance dots: figure 5's maps carry none, so the two rows of maps match.
plot_coherence_spectrum(ax_complex, significance_markers=False)   # |<C_xy>|: magnitude last

# The symbol alone on each y axis. Both are RMS coherence magnitudes over the channels,
# which is caption material, and the estimators are told apart by the order of <> and ||.
ax_magnitudes.set_ylabel(r"Channel-wise RMS of $\langle|C_{xy}|\rangle$")
ax_complex.set_ylabel(r"Channel-wise RMS of $|\langle C_{xy}\rangle|$")
# Titles name the order of operations, which is the only thing separating the two panels.
# Padded as figure 5's title is, to clear the topographies' band labels.
ax_magnitudes.set_title(f"Average of coherence magnitudes (over participants)", pad=16)
ax_complex.set_title("Magnitude of averaged complex coherence", pad=16)
ax_magnitudes.tick_params(axis="x", labelbottom=False)
ax_complex.set_xlabel("Frequency (Hz)")
stack_fig.align_ylabels([ax_magnitudes, ax_complex])

for ax, label in [(ax_magnitudes, "a)"), (ax_complex, "b)")]:
    ax.text(-0.11, 1.04, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

stem = (f"subject_{SUBJECT_COMBINATION}_vs_grand_average_{PRIMARY_REFERENCE}"
        f"{'_excess' if SUBTRACT_NOISE_FLOOR else ''}"
        f"{'' if len(shown_bands) == len(BANDS) else f'_{len(shown_bands)}bands'}")
stack_fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
stack_fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

for trace, median, name in [(subject_spectrum, subject_null_median, "<|Cxy|>"),
                            (spectrum, null_median, "|<Cxy>|")]:
    print(f"{name}: peak {trace[1:].max():.3f} at "
          f"{frequencies[1:][np.argmax(trace[1:])]:.1f} Hz, null median "
          f"{np.median(median):.4f}")

In [ ]:
ratio = np.abs(coherence)/subject_coherence

In [ ]:
plt.plot(frequencies[1:], ratio[1:], lw=1, alpha=0.2)
plt.plot(frequencies[1:], np.mean(ratio[1:], axis=1), color="red", lw=1.5)
plt.xscale("log")

In [ ]:
plot_topomap(
    np.mean(ratio[(frequencies>140) & (frequencies<200)], axis=0),
    RAW_INFO,
    contours=0
)

## Figure 3 -- lateralisation by sign-flip permutation

**The index** is the plain difference d = |coh|_left - |coh|_right for each subject,
band and left/right pair: the same pairs as figure 3, 27 of them, with the midline
left out. It is formed on the raw band magnitudes, without the noise floor removed and
without dividing by anything. The floor is almost identical on the two sides, so it
adds variance to d but no bias. The ratio of figure 3 has a denominator that reaches
zero once the floor is subtracted: under the average reference it is <= 0 in 1.9% of
subject x pair cells, it reaches |LI| = 514, and that is where its excess kurtosis of
~13 comes from. d has a kurtosis of ~1.3.

**The null** is that each subject's map is mirror-symmetric in distribution: swapping
every left channel with its right partner leaves that subject's map as likely as
before. Mirroring a subject flips the sign of all of its d at once, every pair and
every band together. One permutation is therefore one random sign per subject, and
`SIGN_FLIP_N` of them build the null. Because a whole subject is flipped as a unit, the
correlation between neighbouring pairs and between bands is kept intact.

**The correction** is chosen by `SIGN_FLIP_CORRECTION`, over the family of all 27 pairs x
5 bands = 135 tests. Every option starts from the same permutations. Each pair's
uncorrected p is where its observed mean left - right falls in its *own* null: the
fraction of flips with a |mean| at least as large, (1 + count) / (N + 1), counting the
observed data as one of the permutations. The null is symmetric, so this is the
two-sided 2 x min(q, 1 - q) of the percentile q.

* `"max"` (the default): the maximum statistic. For every permutation the largest
  |statistic| over all 135 tests is kept, and a pair's p is the fraction of those
  maxima at least as large as its own value. This controls the family-wise error (the
  chance of any false pair) exactly, and it uses the correlation between neighbouring
  pairs and between bands, so its threshold is lower than Bonferroni's.
* `"bonferroni"`: uncorrected p < alpha / 135. It controls the same family-wise error
  but assumes the tests are independent, so it is more conservative. Under the average
  reference at alpha = 0.05 it drops the two 4-15 Hz pairs and the borderline 80-120 Hz
  P7/P8 that the max statistic keeps.
* `"fdr_bh"` / `"fdr_by"`: Benjamini-Hochberg or Benjamini-Yekutieli on the
  uncorrected p. These control the expected *fraction* of marked pairs that are false,
  not the chance of any, so they mark more. BH assumes the positive dependence these
  pairs have, and BY holds under any dependence. At q = 0.05, BH spreads over most of
  the low-delta map and into 30-60 and 80-120 Hz, where the power-asymmetry caveat
  below applies.

`SIGN_FLIP_ALPHA` sets the level for all four. The permutation count limits how small a
p can be: 1/(N + 1). Bonferroni at alpha = 0.05 needs p < 3.7e-4, so it needs at least
about 2,700 permutations. At alpha = 0.01 it needs 7.4e-5, which takes about 14,000, and
in practice more. The cell warns when the chosen combination cannot mark anything.

**No t-test or Wilcoxon.** The p comes only from the permutations: the observed mean
left - right against the same mean computed on every flipped dataset. No t, normal or
signed-rank distribution is consulted. For the max correction the mean is divided by
its sign-flip SD, sqrt(sum d^2) / S, so the 135 tests share one scale. No flip changes
that SD, so for each pair on its own the p is unchanged.

**The colour is display only.** `SIGN_FLIP_EFFECT` picks what the lower row draws, and it
does not change any p or any marker. `"rank_biserial"` is the matched rank-biserial r,
(sum of positive ranks - sum of negative ranks) / sum of ranks: +1 means every subject
is left > right. `"d_z"` is Cohen's d_z = mean / SD. Both are scale-free, so bands whose
d differ tenfold in coherence units share one colour scale. The row above it is figure 2's topography itself, the band RMS of the
pooled complex coherence, so the asymmetry is read against the map the reader has
already seen.

**A caveat at high frequencies.** A noisier channel has lower coherence. So a
left/right difference in EEG *power* produces a coherence asymmetry that is not about
tracking. At 80-120 Hz under the average reference, PO7 carries 1.3 dB more power than
PO8 (p = 0.001). That power asymmetry predicts the coherence asymmetry across subjects
(rho = -0.51), and regressing it out takes PO7/PO8 from p = 1e-4 to 0.008 (see
`playground/lateralisation_sign_flip.ipynb`). A high-frequency result here should be
read with that in mind. 0.2-2 Hz has not yet been checked the same way.

In [ ]:
# Sign-flip permutation test on the per-subject left - right difference. The magnitudes
# are figure 3's own cached arrays, so nothing is re-pooled; only the index and the test
# change.
from scipy.stats import rankdata

SIGN_FLIP_N = 50_000            # permutations; the smallest attainable p is 1 / (N + 1)
SIGN_FLIP_SEED = 0

# The multiple-comparison correction over the 27 pairs x 5 bands, and its level. "max" is
# the maximum statistic, which controls the family-wise error and uses the correlation
# between tests. "bonferroni" applies alpha / 135 to the uncorrected permutation p.
# "fdr_bh" and "fdr_by" control the false discovery rate instead, and mark more.
SIGN_FLIP_CORRECTION = "bonferroni"   # "max" | "bonferroni" | "fdr_bh" | "fdr_by"
SIGN_FLIP_ALPHA = ALPHA

# What the lower row of the figure draws. Display only: the test is the same whichever is
# chosen. Both are scale-free, so bands whose d differ tenfold share one colour scale.
SIGN_FLIP_EFFECT = "d_z"    # "rank_biserial" | "d_z"
SIGN_FLIP_EFFECT_LABEL = {"rank_biserial": "Rank-biserial r",
                          "d_z": "Cohen's $d$"}[SIGN_FLIP_EFFECT]

# Every subject a reference has (81 for avg, 78 for linked mastoids), not figure 3's
# common 78: the average reference is the one this figure is drawn for, and with all of
# its subjects the test runs on the same people as the figure 2 map above it. Set True
# to cut every reference to the subjects all three share.
SIGN_FLIP_COMMON_SUBJECTS = False


def flipped_mean(d, signs):
    """Mean of ``d`` (S, K) over subjects under each row of ``signs`` (N, S), standardised.

    Divided by the sign-flip null's own SD, sqrt(sum d^2) / S, which no flip changes. For a
    single test that is a constant, so its percentile -- and its p -- is exactly the raw
    mean left - right's. It matters only for the max correction, where 135 tests share one
    threshold and must be on one scale.
    """
    return (signs @ d) / np.sqrt((d ** 2).sum(0))


def effect_sizes(d):
    """Both display effect sizes per column of ``d`` (S, K)."""
    ranks = rankdata(np.abs(d), axis=0) * (d != 0)
    return {"rank_biserial": (ranks * np.sign(d)).sum(0) / ranks.sum(0),
            "d_z": d.mean(0) / d.std(0, ddof=1)}


SIGN_FLIP_CORRECTION_LABEL = {
    "max": "max-statistic FWER", "bonferroni": "Bonferroni-corrected permutation",
    "fdr_bh": "Benjamini-Hochberg FDR", "fdr_by": "Benjamini-Yekutieli FDR",
}[SIGN_FLIP_CORRECTION]


def sign_flip_significant(p_uncorrected, p_max, correction=SIGN_FLIP_CORRECTION,
                          alpha=SIGN_FLIP_ALPHA):
    """Which tests survive the chosen correction; the family is the arrays passed in."""
    if correction == "max":
        return p_max < alpha
    if correction == "bonferroni":
        return p_uncorrected < alpha / p_uncorrected.size
    method = {"fdr_bh": "bh", "fdr_by": "by"}[correction]
    flat = false_discovery_control(p_uncorrected.ravel(), method=method)
    return flat.reshape(p_uncorrected.shape) < alpha


# The strictest threshold each correction applies to a single test. If the smallest p
# the permutations can produce is not below it, nothing can be marked, whatever the data.
n_sign_flip_tests = n_pairs * n_li_bands
strictest = {
    "max": SIGN_FLIP_ALPHA,
    "bonferroni": SIGN_FLIP_ALPHA / n_sign_flip_tests,
    "fdr_bh": SIGN_FLIP_ALPHA / n_sign_flip_tests,
    "fdr_by": SIGN_FLIP_ALPHA / (n_sign_flip_tests
                                 * (1 / np.arange(1, n_sign_flip_tests + 1)).sum()),
}[SIGN_FLIP_CORRECTION]
if 1 / (SIGN_FLIP_N + 1) >= strictest:
    print(f"WARNING: {SIGN_FLIP_N} permutations give a smallest p of "
          f"{1 / (SIGN_FLIP_N + 1):.1e}, not below {SIGN_FLIP_CORRECTION}'s strictest "
          f"threshold of {strictest:.1e}; raise SIGN_FLIP_N.\n")


def sign_flip_lateralisation(reference, bands=LATERALISATION_BANDS):
    """Sign-flip test of the mean left - right per pair and band.

    Returns the mean d, its uncorrected and max-statistic p, which pairs survive
    SIGN_FLIP_CORRECTION, and both effect sizes, each ``(n_bands, n_pairs)``, plus the
    hemisphere-mean p per band.
    """
    observed, _, _, subjects = subject_band_topographies(reference, bands)
    if SIGN_FLIP_COMMON_SUBJECTS:
        keep = [i for i, name in enumerate(subjects) if name in set(COMMON_SUBJECTS)]
        observed, subjects = observed[keep], [subjects[i] for i in keep]
    pairs = channel_pairs(info_for(reference))
    left = observed[:, :, [pair[0] for pair in pairs]]
    right = observed[:, :, [pair[1] for pair in pairs]]
    d = (left - right).astype(np.float64)                     # (S, n_bands, n_pairs)
    shape = d.shape[1:]
    flat = d.reshape(len(d), -1)

    # One sign per subject per permutation, shared by every pair and band -- a subject
    # is mirrored whole or not at all.
    rng = np.random.default_rng(SIGN_FLIP_SEED)
    signs = rng.choice([-1.0, 1.0], size=(SIGN_FLIP_N, len(d)))
    identity = np.ones((1, len(d)))

    value = np.abs(flipped_mean(flat, identity)[0])
    null = np.abs(flipped_mean(flat, signs))                      # (N, n_tests)
    # Two-sided: |mean| against the |null|, which is 2 x the smaller tail because the
    # sign-flip null is symmetric. The observed data are one of the permutations, hence
    # the + 1 on both sides. Uncorrected: each test against its own null. Max: against
    # the null maxima over all 135 tests.
    p_uncorrected = ((1 + (null >= value).sum(0)) / (SIGN_FLIP_N + 1)).reshape(shape)
    p_max = ((1 + (null.max(1)[None] >= value[:, None]).sum(1))
             / (SIGN_FLIP_N + 1)).reshape(shape)

    hemisphere = d.mean(-1)                                       # (S, n_bands)
    hemisphere_value = np.abs(flipped_mean(hemisphere, identity)[0])
    hemisphere_p = ((1 + (np.abs(flipped_mean(hemisphere, signs)) >= hemisphere_value).sum(0))
                    / (SIGN_FLIP_N + 1))

    return dict(pairs=pairs, subjects=subjects, d=d, mean=d.mean(0),
                p_uncorrected=p_uncorrected, p_max=p_max,
                significant=sign_flip_significant(p_uncorrected, p_max),
                effect={name: values.reshape(shape)
                        for name, values in effect_sizes(flat).items()},
                hemisphere_p=hemisphere_p)


SIGN_FLIP_RESULTS = {reference: sign_flip_lateralisation(reference) for reference in REFERENCES}

print(f"mean left - right, {SIGN_FLIP_N} sign flips, {SIGN_FLIP_CORRECTION_LABEL} "
      f"< {SIGN_FLIP_ALPHA:g} over {n_pairs} pairs x {n_li_bands} bands; subjects: "
      + ", ".join(f"{reference} {len(result['subjects'])}"
                  for reference, result in SIGN_FLIP_RESULTS.items()))
print(f"pairs listed with {SIGN_FLIP_EFFECT_LABEL} and uncorrected p; hemisphere p is the "
      "mean d over all pairs, one test per band, uncorrected\n")
for reference, result in SIGN_FLIP_RESULTS.items():
    labels = [pair[2] for pair in result["pairs"]]
    effect = result["effect"][SIGN_FLIP_EFFECT]
    print(f"{REFERENCE_LABELS[reference]}:")
    for index, (low, high) in enumerate(LATERALISATION_BANDS):
        marked = np.flatnonzero(result["significant"][index])
        order = marked[np.argsort(result["p_uncorrected"][index, marked])]
        print(f"  {low:5.1f}-{high:5.1f} Hz  {len(marked):2d}/{n_pairs} pairs   "
              f"hemisphere p = {result['hemisphere_p'][index]:.3f}")
        if len(order):
            print("      " + ", ".join(
                f"{labels[i]} {effect[index, i]:+.2f} (p {result['p_uncorrected'][index, i]:.0e})"
                for i in order))

In [ ]:
# Figure 3's layout with the sign-flip result in the lower row: Figure 2's own coherence
# topography above, the chosen effect size below, spread antisymmetrically
# over the scalp (+value on the left channel, -value on its partner, midline zero).


def figure_2_topographies(reference, bands=LATERALISATION_BANDS):
    """Figure 2's band maps ``(n_bands, n_eeg)``: complex coherence pooled first, magnitude after.

    The band RMS of the duration-weighted grand average -- the statistic figure 2 draws --
    not figure 3's median of per-subject magnitudes, which takes the magnitude first and
    so keeps each subject's noise floor and loses the phase cancellation between them.
    It is pooled over every subject that reference has (81 for avg), as figure 2 is,
    which is also the test's subject set unless SIGN_FLIP_COMMON_SUBJECTS is True.
    """
    pooled = pooled_grand_average(reference)
    masks, _ = band_bin_masks(pooled["frequencies"], bands)
    power = np.abs(pooled["coherence"]) ** 2
    return np.stack([np.sqrt(power[mask].mean(0)) for mask in masks])


FIGURE_2_TOPOGRAPHIES = {reference: figure_2_topographies(reference)
                         for reference in LI_REFERENCES}


def sign_flip_on_scalp(reference, band_index, effect=SIGN_FLIP_EFFECT):
    """Effect sizes on the sensors and the mask of sensors in a surviving pair."""
    result = SIGN_FLIP_RESULTS[reference]
    values = np.zeros(len(info_for(reference).ch_names), float)
    mask = np.zeros(len(values), bool)
    for pair_index, (left, right, _) in enumerate(result["pairs"]):
        values[left] = result["effect"][effect][band_index, pair_index]
        values[right] = -values[left]
        if result["significant"][band_index, pair_index]:
            mask[left] = mask[right] = True
    return values, mask


# One symmetric scale for every effect map, to the next 0.1 above the largest drawn, over
# the references actually drawn.
SIGN_FLIP_LIMIT = np.ceil(10 * max(np.abs(sign_flip_on_scalp(reference, band)[0]).max()
                                   for reference in LI_REFERENCES
                                   for band in range(n_li_bands))) / 10
SIGN_FLIP_ROW_KINDS = ("coherence", "asymmetry")

available = 1 - LI_LABEL_ROOM - LI_BAR_ROOM
cell_size = available * LI_WIDTH / n_li_bands                       # inches, square
grid_height = len(SIGN_FLIP_ROW_KINDS) * len(LI_REFERENCES) * cell_size
figure_height = grid_height + LI_TOP_MARGIN + LI_BOTTOM_MARGIN

fig = plt.figure(figsize=(LI_WIDTH, figure_height))
outer = fig.add_gridspec(len(LI_REFERENCES), 1, hspace=LI_GROUP_GAP,
                         left=LI_LABEL_ROOM, right=LI_LABEL_ROOM + available,
                         top=1 - LI_TOP_MARGIN / figure_height,
                         bottom=LI_BOTTOM_MARGIN / figure_height)

effect_row_axis = None
for block, reference in enumerate(LI_REFERENCES):
    info = info_for(reference)
    inner = outer[block, 0].subgridspec(len(SIGN_FLIP_ROW_KINDS), n_li_bands,
                                        hspace=LI_PAIR_GAP, wspace=LI_PAIR_GAP)
    for band in range(n_li_bands):
        ax_coherence = fig.add_subplot(inner[0, band])
        plot_topomap(FIGURE_2_TOPOGRAPHIES[reference][band], info, cmap="Reds",
                     contours=0, sensors=False, show=False, axes=ax_coherence)

        ax_effect = effect_row_axis = fig.add_subplot(inner[1, band])
        values, mask = sign_flip_on_scalp(reference, band)
        plot_topomap(values, info, cmap=LI_CMAP, contours=0, sensors=False,
                     vlim=(-SIGN_FLIP_LIMIT, SIGN_FLIP_LIMIT), mask=mask,
                     mask_params=LI_MARK, show=False, axes=ax_effect)

        if block == 0:
            box = ax_coherence.get_position()
            low, high = LATERALISATION_BANDS[band]
            low_label = f"{low:.1f}" if low < 1 else f"{int(low)}"
            fig.text((box.x0 + box.x1) / 2, box.y1 + 0.012,
                     f"{low_label}-{int(high)} Hz", ha="center", va="bottom",
                     fontsize=TICK_FONTSIZE)

        if band == 0:
            for axis, kind in zip((ax_coherence, ax_effect), SIGN_FLIP_ROW_KINDS):
                box = axis.get_position()
                fig.text(LI_LABEL_ROOM - 0.015, (box.y0 + box.y1) / 2, kind,
                         ha="right", va="center", fontsize=TICK_FONTSIZE)

    if len(LI_REFERENCES) > 1:
        box = outer[block, 0].get_position(fig)
        fig.text(0.012, (box.y0 + box.y1) / 2, REFERENCE_LABELS[reference],
                 ha="left", va="center", rotation=90, fontsize=DEFAULT_FONTSIZE)

# Hung off the effect row's real box, as in figure 3, so it keys that row alone.
row_box = effect_row_axis.get_position()
cax = fig.add_axes([LI_LABEL_ROOM + available + 0.035, row_box.y0, 0.018,
                    row_box.height])
bar = fig.colorbar(ScalarMappable(Normalize(-SIGN_FLIP_LIMIT, SIGN_FLIP_LIMIT), LI_CMAP),
                   cax=cax, orientation="vertical")
bar.set_ticks([-SIGN_FLIP_LIMIT, 0, SIGN_FLIP_LIMIT])
bar.set_label(f"{SIGN_FLIP_EFFECT_LABEL}\n(right > left)",
              fontsize=TICK_FONTSIZE, linespacing=1.4)
bar.ax.tick_params(labelsize=TICK_FONTSIZE)
bar.outline.set_visible(False)

stem = (f"lateralisation_sign_flip_{'_'.join(LI_REFERENCES)}_{SIGN_FLIP_EFFECT}"
        f"_{SIGN_FLIP_CORRECTION}_{SIGN_FLIP_ALPHA:g}_{SIGN_FLIP_N}_{'common' if SIGN_FLIP_COMMON_SUBJECTS else 'all'}_subjects")
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

## Combined figure -- the grand average and the cochleogram

Figure 1's two panels with the subband coherence underneath, so the broadband result and the
cochlear axis it collapses are read together rather than in separate figures.

* **(a)** and **(b)** are figure 1 exactly -- the same `plot_coherence_spectrum` and
  `plot_unit_rates`, so nothing can drift between the two figures.
* **(c)** and **(d)** are the grand average resolved over the 60 gammatone subbands, from
  `get_referenced_coherence.py`'s pooled real coherence. This is the same estimator (a
  duration-weighted complex average, magnitude taken at the end) and the same frequency grid, one
  axis wider.

The two subband panels average over **different sensor sets**, which is the arrangement the
original code used and is deliberate: below 30 Hz tracking is broad over the scalp, so (c) takes
the whole montage; above it the response is focal over the midline -- as figure 1's own 80-120 and
140-250 Hz topographies show -- so (d) takes the central cluster. A whole-montage average up
there would dilute a focal response with 49 channels that do not carry it.

In [ ]:
from scipy.ndimage import uniform_filter1d

# The pooled real subband coherence. One directory holds every reference -- the files are named
# by reference rather than filed under it -- and `get_referenced_coherence.py` computes all three
# in one ~25 min pass. The 500-draw subband null that `get_null_cochleogram_coherence.py` would
# also produce costs ~40 GB and ~10 h per reference and is not read here, so it is not run.
SUBBAND_AVERAGE = (Path(CONFIG["coherence_dir"]) / "reference_average"
                   / f"average_coherence_{PRIMARY_REFERENCE}.npz")
FORMANT_DIR = Path(CONFIG["speech_units_dir"]) / "formants"
N_SUBBANDS = 60

with np.load(SUBBAND_AVERAGE) as saved:
    if "subband_coherence" not in saved.files:
        raise KeyError(f"{SUBBAND_AVERAGE.name} predates the subband arrays; re-run "
                       "scripts/coherence/get_referenced_coherence.py")
    subband_coherence = saved["subband_coherence"]            # (F, 60, n_eeg)
    centre_frequencies = saved["centre_frequencies"]
    subband_channels = [str(name) for name in saved["channel_names"]]
    subband_trials = int(saved["n_trials"])
    subband_hours = float(saved["total_duration"]) / 3600
    # Both axes checked rather than assumed: the panels index the subband axis by position, and
    # share this notebook's frequency axis with panels (a) and (b).
    assert np.allclose(saved["frequencies"], frequencies), "different coherence frequency axis"

assert subband_coherence.shape[1] == N_SUBBANDS, f"expected {N_SUBBANDS} subbands"
assert np.all(np.diff(centre_frequencies) > 0), "subbands not in ascending centre order"
assert subband_channels == list(INFO.ch_names), (
    "the subband file's channel axis differs from this notebook's INFO for "
    f"{PRIMARY_REFERENCE}: {len(subband_channels)} vs {len(INFO.ch_names)}"
)

subband_power = np.abs(subband_coherence) ** 2

# --- formant ranges, for the brackets in panel (d) -------------------------------------------
# Each stimulus's F1-F3 histogram weighted by its listening time, and a formant's range is the
# central 99% of the pooled distribution. `stimulus_seconds` is the pooling this notebook already
# uses everywhere else, so the weights match the coherence the brackets are drawn over.
FORMANT_NAMES = ["F1", "F2", "F3"]
FORMANT_COVERAGE = 0.99

missing_formants = [name for name in stimulus_seconds
                    if not (FORMANT_DIR / f"{name}_formants.npz").exists()]
assert not missing_formants, f"no formant file for {missing_formants[:3]}"

pooled_formants = 0.0
for name, seconds in stimulus_seconds.items():
    with np.load(FORMANT_DIR / f"{name}_formants.npz") as saved:
        pooled_formants = (pooled_formants
                           + np.nan_to_num(saved["counts"][:3].astype(float)) * seconds)
        formant_bins = saved["bin_centres"]

formant_cdf = pooled_formants.cumsum(1) / pooled_formants.sum(1, keepdims=True)
_tail = (1 - FORMANT_COVERAGE) / 2
formant_low = formant_bins[np.argmax(formant_cdf >= _tail, axis=1)]
formant_high = formant_bins[np.argmax(formant_cdf >= 1 - _tail, axis=1)]

print(f"{PRIMARY_REFERENCE}: {N_SUBBANDS} subbands, {centre_frequencies[0]:.0f}-"
      f"{centre_frequencies[-1]:.0f} Hz; {subband_trials} trials, {subband_hours:.1f} h, "
      f"{len(subband_channels)} sensors")
for label, low, high in zip(FORMANT_NAMES, formant_low, formant_high):
    print(f"  {label}: {low:.0f}-{high:.0f} Hz")

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.patches import Circle

# --- what each subband panel averages ---------------------------------------------------------
# Different sets on purpose. Below 30 Hz tracking is broad over the scalp; above it, it is focal
# over the midline (figure 1's 80-120 and 140-250 Hz topographies), and a 64-channel average
# there would dilute it with channels that do not carry it.
SUBBAND_ALL_SENSORS = list(INFO.ch_names)
SUBBAND_CENTRAL_CLUSTER = [
    "F1", "Fz", "F2",
    "FC1", "FCz", "FC2",
    "C1", "Cz", "C2",
    "CP1", "CPz", "CP2",
    "P1", "Pz", "P2",
]


def subband_sensors(sensors, label):
    """The named sensors this reference actually carries, reporting anything missing.

    FCz is in the central cluster and is dropped by its own reference, so the cluster is
    intersected rather than assumed -- and the loss is printed rather than silently averaging a
    different cluster from the one the caption claims.
    """
    kept = [name for name in sensors if name in subband_channels]
    missing = sorted(set(sensors) - set(kept))
    if missing:
        print(f"{label}: {len(kept)} of {len(sensors)} sensors "
              f"({', '.join(missing)} not in the {PRIMARY_REFERENCE} reference)")
    return kept


PANEL_C_SENSORS = subband_sensors(SUBBAND_ALL_SENSORS, "panel (c)")
PANEL_D_SENSORS = subband_sensors(SUBBAND_CENTRAL_CLUSTER, "panel (d)")

# Panel (d)'s smoothing along frequency, in bins of the 0.2 Hz grid.
SUBBAND_SMOOTHING_BINS = 25

# Both panels on a continuous scale, fixed rather than autoscaled, so a panel is comparable
# between references rather than rescaled to whichever one is drawn. The ends of each range are
# the ladder's ends; the ladder itself survives as the colourbar's ticks, which keeps the round
# numbers a default locator would not pick for a range ending at 0.040.
#
# The two ranges differ by roughly a factor of four, which is how much more coherence the low
# regime carries; one shared range would flatten (d) entirely.
LOW_TICKS = np.arange(0.0, 0.0401, 0.005)
# Even 0.001 steps. `np.linspace(0, 0.009, 9)` steps by 0.001125, so its labels rounded to
# 0.000, 0.001, 0.002, 0.003, 0.005 ... -- the printed numbers were not the tick values.
HIGH_TICKS = np.arange(0.0, 0.0091, 0.001)
SUBBAND_CMAP = "plasma"

CENTRE_TICKS = [50, 100, 1000, 8000]
# Formant brackets: x position, and the height of each label. Further left than a panel with
# no head would need -- the rightmost bracket has to stop short of the inset, whose left edge is
# at 0.71 of the axes, i.e. about 219 Hz on (d)'s 20-300 Hz scale. The labels overhang that edge
# but sit well above the inset's top, at 0.28.
FORMANT_X = [210, 235, 260]
FORMANT_LABEL_Y = [490, 1400, 2700]


def subband_rms(sensors):
    """RMS coherence magnitude over the named sensors, per frequency and subband: (F, 60)."""
    index = [INFO.ch_names.index(name) for name in sensors]
    return np.sqrt(subband_power[..., index].mean(-1))


def nan_boxcar(values, size):
    """Running mean along frequency that skips NaNs.

    The mains bins are set to NaN before smoothing rather than after: a boxcar over a line
    artefact spreads it into its neighbours, and removing the bins afterwards would leave that
    contamination sitting just outside the notch. Dropped bins are filled from their neighbours.
    """
    valid = ~np.isnan(values)
    total = uniform_filter1d(np.where(valid, values, 0.0), size, axis=0, mode="nearest")
    count = uniform_filter1d(valid.astype(float), size, axis=0, mode="nearest")
    return total / count


def scale_colorbar(mesh, ax, ticks, label_every=1):
    """A colourbar with a square extension cap, labelled at every `label_every` tick.

    The scale itself is continuous; these ticks are only where it gets numbers, and they are the
    ladder the banded version used, so the readings do not change.

    `extendrect` squares off the over-range block, which the default draws as a triangle -- an
    arrow pointing out of the scale rather than more of it.
    """
    colorbar = plt.colorbar(mesh, ax=ax, pad=0.02, extend="max", extendrect=True)
    ticks = np.asarray(ticks)[::label_every]
    colorbar.set_ticks(ticks, labels=[f"{tick:.3f}" for tick in ticks])
    colorbar.ax.tick_params(labelsize=TICK_FONTSIZE)
    return colorbar


# Both heads sit in the same corner, so the pair reads as one legend in two places rather than
# as two arbitrary placements. The bracket positions below are what moves to make room in (d).
SENSOR_INSET = [0.71, -0.006, 0.30, 0.30]


def sensor_inset(ax, sensors, rect):
    """A small head marking the sensors the panel averages.

    The two panels average different sets, so each has to say which -- a caption alone would
    leave a reader to work out from the text which plate the central cluster belongs to.
    """
    inset = ax.inset_axes(rect)
    # The head sits over the heatmap, so it gets a white disc to read against.
    inset.add_patch(Circle((0.50, 0.45), 0.5, transform=inset.transAxes,
                           facecolor="white", edgecolor="none", zorder=0))
    plot_topomap(
        np.zeros(len(INFO["ch_names"])), INFO, axes=inset, show=False, sensors=False,
        cmap="Greys", vlim=(0, 1), contours=0,
        mask=np.array([name in sensors for name in INFO["ch_names"]]),
        mask_params=dict(marker="o", markerfacecolor=INK, markeredgecolor=INK,
                         linewidth=0, markersize=1.5),
    )


def centre_frequency_axis(ax, label=True):
    ax.set_yscale("log")
    ax.set_yticks(CENTRE_TICKS, CENTRE_TICKS if label else [""] * len(CENTRE_TICKS))
    if label:
        ax.set_ylabel("Cochlear centre frequency (Hz)")
    ax.set_xlabel("Frequency for coherence (Hz)")
    bare(ax)


def plot_low_modulation(ax, sensors):
    """Panel (c): 0.1-30 Hz, unsmoothed, discrete scale, whole montage."""
    norm = Normalize(vmin=LOW_TICKS[0], vmax=LOW_TICKS[-1])
    mesh = ax.pcolormesh(frequencies[1:], centre_frequencies, subband_rms(sensors)[1:].T,
                         shading="auto", cmap=SUBBAND_CMAP, norm=norm, rasterized=True)
    ax.set_xlim(0.1, 30)
    ax.set_xticks(range(5, 31, 5), range(5, 31, 5))
    centre_frequency_axis(ax)
    scale_colorbar(mesh, ax, LOW_TICKS)
    sensor_inset(ax, sensors, SENSOR_INSET)


def plot_high_modulation(ax, sensors):
    """Panel (d): 30-300 Hz, mains dropped, smoothed, discrete scale, central cluster."""
    values = subband_rms(sensors)
    harmonics = np.arange(LINE_FREQ, frequencies[-1] + LINE_FREQ, LINE_FREQ)
    line = (np.abs(frequencies[:, None] - harmonics) <= NOTCH_HALFWIDTH).any(1)
    values[line] = np.nan
    values = nan_boxcar(values, SUBBAND_SMOOTHING_BINS)

    norm = Normalize(vmin=HIGH_TICKS[0], vmax=HIGH_TICKS[-1])
    mesh = ax.pcolormesh(frequencies[1:], centre_frequencies, values[1:].T, shading="auto",
                         cmap=SUBBAND_CMAP, norm=norm, rasterized=True)
    ax.set_xlim(20, 300)
    ax.set_xticks(range(50, 301, 50), range(50, 301, 50))
    # The shared cochlear axis is labelled once, on panel (c).
    centre_frequency_axis(ax, label=False)

    for x, low, high, name, label_y in zip(FORMANT_X, formant_low, formant_high,
                                           FORMANT_NAMES, FORMANT_LABEL_Y):
        ax.plot([x, x], [low, high], c="white", lw=2)
        ax.plot([x - 2, x + 2], [low, low], c="white", lw=2)
        ax.plot([x - 2, x + 2], [high, high], c="white", lw=2)
        ax.text(x + 15, label_y, name, color="white", fontsize=TICK_FONTSIZE,
                ha="center", weight="bold")

    scale_colorbar(mesh, ax, HIGH_TICKS)
    sensor_inset(ax, sensors, SENSOR_INSET)

In [ ]:
# Taller than figure 1 by the subband row. Same 6.27 in text width, so every figure in the
# document carries text at the same physical size.
COMBINED_FIGSIZE = (6.27, 7.5)

fig = plt.figure(figsize=COMBINED_FIGSIZE)
# Nested rather than one 3x2 grid, so the a-b gap and the b-c gap are set separately. They want
# different things: (a) and (b) share an x axis and sit close, as in figure 1; (c) and (d) start
# a new pair of axes under (b)'s x-label and need clear air above them.
outer = fig.add_gridspec(2, 1, height_ratios=[17, 8], hspace=0.3)
top = outer[0].subgridspec(2, 1, height_ratios=[10, 4], hspace=0.28)
# The gutter has to hold (c)'s colourbar and its tick labels as well as separate the two plates
# -- a colourbar takes its space out of its own panel's box, not out of this gap.
bottom = outer[1].subgridspec(1, 2, wspace=0.30)

ax_spectrum = fig.add_subplot(top[0])
ax_rates = fig.add_subplot(top[1], sharex=ax_spectrum)
ax_low = fig.add_subplot(bottom[0])
ax_high = fig.add_subplot(bottom[1])

ax_spectrum.tick_params(axis="x", labelbottom=False)
ax_rates.tick_params(axis="x", labelbottom=True)

plot_coherence_spectrum(ax_spectrum)
plot_unit_rates(ax_rates)
plot_low_modulation(ax_low, PANEL_C_SENSORS)
plot_high_modulation(ax_high, PANEL_D_SENSORS)

ax_spectrum.set_title("EEG-envelope coherence across the spectrum")
ax_rates.set_title("Rates of speech units in the stimuli")
ax_low.set_title("Low modulation frequencies")
ax_high.set_title("High modulation frequencies")

# (a) and (b) span the figure, so their markers share an x; (c) and (d) each carry their own
# left margin -- (c)'s holds a y-label the marker has to clear -- so theirs are placed per axes.
# (c) and (d) also sit higher than their titles rather than beside them: a colourbar takes
# its space out of its own panel's box, so those two axes are narrow enough that a centred
# title reaches almost to the left spine.
for ax, label, x, y in [(ax_spectrum, "a)", -0.075, 1.0), (ax_rates, "b)", -0.075, 1.05),
                        (ax_low, "c)", -0.2, 1.06), (ax_high, "d)", -0.23, 1.06)]:
    ax.text(x, y, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

# No suptitle: the reference, trial count and listening time belong in the caption, as in the
# other figures. The two load cells print them.
stem = f"grand_average_and_subbands_{PRIMARY_REFERENCE}"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()

## Supplementary -- the subband coherence under two references

The same two plates as panels (c) and (d) above, drawn once per reference: average on the top
row, linked mastoids on the bottom. All four average the **whole 64-channel montage**, including
the two high-frequency panels -- this figure is asking whether the picture survives a change of
reference, not where on the scalp it lives, and a fixed sensor set is what makes the two rows
comparable. Only (a) carries the head, since all four sets are identical.

Both columns keep the ladders panels (c) and (d) use, so a colour means the same coherence in
every panel here and in the figure above. That is the whole point: the rows can only be compared
if neither is rescaled to itself.

Linked mastoids is the reference worth checking. It subtracts a single pair of electrodes rather
than the montage mean, which makes it sensitive to anything those two channels pick up -- the
lateralisation analysis above found its band magnitudes correlate with laterality at rho = 0.78,
against 0.56 for the average. If the cochlear structure here were an artefact of referencing, this
is where it would show.

In [ ]:
# Both references read through one loader, so nothing can differ between the rows except the
# file. The average reference is not re-read from the cells above -- those hold whichever
# reference PRIMARY_REFERENCE names, and this figure hard-codes its two.
SUPP_REFERENCES = ["avg", "linked_mastoids"]
SUPP_REFERENCE_LABELS = {"avg": "Average reference", "linked_mastoids": "Linked mastoids"}


def load_subband_reference(reference):
    """The pooled real subband coherence for one reference, with the info to plot it.

    Returns power rather than coherency: every panel takes a magnitude, and squaring once here
    keeps the two rows on identical arithmetic.
    """
    path = (Path(CONFIG["coherence_dir"]) / "reference_average"
            / f"average_coherence_{reference}.npz")
    with np.load(path) as saved:
        if "subband_coherence" not in saved.files:
            raise KeyError(f"{path.name} predates the subband arrays; re-run "
                           "scripts/coherence/get_referenced_coherence.py")
        coherence = saved["subband_coherence"]                     # (F, 60, n_eeg)
        centres = saved["centre_frequencies"]
        names = [str(name) for name in saved["channel_names"]]
        record = dict(reference=reference, power=np.abs(coherence) ** 2,
                      centre_frequencies=centres, channel_names=names,
                      n_trials=int(saved["n_trials"]),
                      hours=float(saved["total_duration"]) / 3600)
        # The frequency axis has to be this notebook's: the panels index it, and the mains notch
        # below is built from it.
        assert np.allclose(saved["frequencies"], frequencies), \
            f"{reference}: different coherence frequency axis"

    assert coherence.shape[1] == N_SUBBANDS, f"{reference}: expected {N_SUBBANDS} subbands"
    assert np.all(np.diff(centres) > 0), f"{reference}: subbands not in ascending centre order"
    # `info_for` knows what the pipeline drops per reference; check the file agrees rather than
    # trusting it, since the topography in (a) indexes this axis.
    record["info"] = info_for(reference)
    assert list(record["info"].ch_names) == names, (
        f"{reference}: file has {len(names)} channels, info_for gives "
        f"{len(record['info'].ch_names)}")
    return record


supp = {reference: load_subband_reference(reference) for reference in SUPP_REFERENCES}

# What each row reaches, against the top of its ladder -- the extension block is only honest if
# we know whether it is carrying anything.
for reference, record in supp.items():
    rms = np.sqrt(record["power"].mean(-1))                        # (F, 60), all channels
    low = rms[(frequencies > 0.1) & (frequencies <= 30)].max()
    high = rms[(frequencies >= 20) & (frequencies <= 300)].max()
    print(f"{reference:16s} {record['n_trials']} trials, {record['hours']:.1f} h, "
          f"{len(record['channel_names'])} ch | max RMS: low {low:.4f} "
          f"(scale tops at {LOW_TICKS[-1]:.3f}), high {high:.4f} "
          f"(tops at {HIGH_TICKS[-1]:.3f})")

In [ ]:
SUPP_FIGSIZE = (6.27, 5.0)
# Where the row's reference name sits, left of the cochlear axis label.
ROW_LABEL_X = -0.52

fig = plt.figure(figsize=SUPP_FIGSIZE)
# The two rows share a column's x axis, so only the bottom one is ticked and labelled; that is
# what lets the rows sit this close without the gap filling with repeated numbers.
grid = fig.add_gridspec(2, 2, hspace=0.24, wspace=0.30)
axes = np.array([[fig.add_subplot(grid[row, col]) for col in range(2)] for row in range(2)])


def supp_rms(record, sensors):
    index = [record["info"].ch_names.index(name) for name in sensors]
    return np.sqrt(record["power"][..., index].mean(-1))           # (F, 60)


def supp_axis(ax, record, label_y, label_x):
    ax.set_yscale("log")
    ax.set_yticks(CENTRE_TICKS, CENTRE_TICKS if label_y else [""] * len(CENTRE_TICKS))
    if label_y:
        ax.set_ylabel("Cochlear centre\nfrequency (Hz)")
    if label_x:
        ax.set_xlabel("Frequency for coherence (Hz)")
    else:
        ax.tick_params(axis="x", labelbottom=False)
    bare(ax)


def supp_low(ax, record, sensors, label_y, label_x, head=False):
    """Left column: 0.1-30 Hz, unsmoothed, on panel (c)'s ladder."""
    norm = Normalize(vmin=LOW_TICKS[0], vmax=LOW_TICKS[-1])
    mesh = ax.pcolormesh(frequencies[1:], record["centre_frequencies"],
                         supp_rms(record, sensors)[1:].T, shading="auto",
                         cmap=SUBBAND_CMAP, norm=norm, rasterized=True)
    ax.set_xlim(0.1, 30)
    ax.set_xticks(range(5, 31, 5), range(5, 31, 5))
    supp_axis(ax, record, label_y, label_x)
    scale_colorbar(mesh, ax, LOW_TICKS)
    if head:
        supp_head(ax, record, sensors)


def supp_high(ax, record, sensors, label_y, label_x, head=False):
    """Right column: 20-300 Hz, mains dropped, smoothed, on panel (d)'s ladder."""
    values = supp_rms(record, sensors)
    harmonics = np.arange(LINE_FREQ, frequencies[-1] + LINE_FREQ, LINE_FREQ)
    line = (np.abs(frequencies[:, None] - harmonics) <= NOTCH_HALFWIDTH).any(1)
    values[line] = np.nan
    values = nan_boxcar(values, SUBBAND_SMOOTHING_BINS)

    norm = Normalize(vmin=HIGH_TICKS[0], vmax=HIGH_TICKS[-1])
    mesh = ax.pcolormesh(frequencies[1:], record["centre_frequencies"], values[1:].T,
                         shading="auto", cmap=SUBBAND_CMAP, norm=norm, rasterized=True)
    ax.set_xlim(20, 300)
    ax.set_xticks(range(50, 301, 50), range(50, 301, 50))
    supp_axis(ax, record, label_y, label_x)

    for x, low, high, name, y in zip(FORMANT_X, formant_low, formant_high,
                                     FORMANT_NAMES, FORMANT_LABEL_Y):
        ax.plot([x, x], [low, high], c="white", lw=2)
        ax.plot([x - 2, x + 2], [low, low], c="white", lw=2)
        ax.plot([x - 2, x + 2], [high, high], c="white", lw=2)
        ax.text(x + 15, y, name, color="white", fontsize=TICK_FONTSIZE, ha="center",
                weight="bold")

    scale_colorbar(mesh, ax, HIGH_TICKS)
    if head:
        supp_head(ax, record, sensors)


def supp_head(ax, record, sensors):
    """`sensor_inset`, but reading the row's own montage rather than the notebook's INFO."""
    info = record["info"]
    inset = ax.inset_axes(SENSOR_INSET)
    inset.add_patch(Circle((0.50, 0.45), 0.5, transform=inset.transAxes,
                           facecolor="white", edgecolor="none", zorder=0))
    plot_topomap(
        np.zeros(len(info["ch_names"])), info, axes=inset, show=False, sensors=False,
        cmap="Greys", vlim=(0, 1), contours=0,
        mask=np.array([name in sensors for name in info["ch_names"]]),
        mask_params=dict(marker="o", markerfacecolor=INK, markeredgecolor=INK,
                         linewidth=0, markersize=1.5),
    )


for row, reference in enumerate(SUPP_REFERENCES):
    record = supp[reference]
    # Every panel averages the whole montage this reference carries -- the question here is the
    # reference, not the sensor set.
    sensors = record["channel_names"]
    bottom_row = row == len(SUPP_REFERENCES) - 1
    supp_low(axes[row, 0], record, sensors, label_y=True, label_x=bottom_row, head=row == 0)
    supp_high(axes[row, 1], record, sensors, label_y=False, label_x=bottom_row)
    axes[row, 0].text(ROW_LABEL_X, 0.5, SUPP_REFERENCE_LABELS[reference],
                      transform=axes[row, 0].transAxes, rotation=90, va="center", ha="center",
                      fontsize=DEFAULT_FONTSIZE, fontweight="bold")

axes[0, 0].set_title("Low modulation frequencies")
axes[0, 1].set_title("High modulation frequencies")

for ax, label in zip(axes.ravel(), ["a)", "b)", "c)", "d)"]):
    ax.text(-0.30, 1.03, label, transform=ax.transAxes, fontsize=PANEL_LABEL_FONTSIZE,
            fontweight="bold", va="bottom", ha="left")

stem = "subband_coherence_by_reference"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()